# HNDSR — Hybrid Neural Operator-Diffusion Super-Resolution
## Databricks + MLflow Edition

**Authors:** Adil Khan, Rakshit Modanwal, Harsh Vardhan, Piyush Jain, Yash Vikram  
**Affiliation:** Department of CSE, Indian Institute of Information Technology, Nagpur  
**Dataset:** 4× Satellite Image Super-Resolution (Kaggle) [Tudela, 2023]

---

### Motivation

High-resolution satellite imagery is critical for environmental monitoring, precision agriculture, disaster assessment, and urban planning. Existing SR approaches have key limitations:

| Approach | Limitation |
|----------|-----------|
| **CNN-based** (SRCNN, EDSR, RCAN) | Pixel-wise L1/L2 losses → over-smoothed textures; fixed integer scale factors only |
| **GAN-based** (SRGAN, ESRGAN) | Unstable training (mode collapse, gradient imbalance); perceptual artifacts in complex scenes |
| **Diffusion-based** (SR3) | Hundreds of denoising steps → high latency; operates in pixel space; fixed 4× scale |
| **E²DiffSR** | Efficient latent diffusion but limited to discrete training scales |
| **NeurOp-Diff** | Continuous-scale via neural operators but pixel-space diffusion → high compute cost |

HNDSR combines the best of both worlds: **E²DiffSR's latent-space efficiency** + **NeurOp-Diff's continuous-scale neural operator prior**.

### Three Key Contributions (from Paper)

1. **Neural Operator Prior** — learns a continuous mapping $N_\phi : \mathbb{R}^{H \times W} \to \mathcal{F}(\Omega)$ from LR inputs to scale-invariant latent representations, enabling flexible non-integer upscaling.
2. **Latent Diffusion Backbone** — a conditioned diffusion model that reconstructs high-frequency textures in a compact latent space with fewer denoising steps → faster inference.
3. **Joint Optimisation Strategy** — a multi-term loss combining content ($\mathcal{L}_{AE}$), latent mapping ($\mathcal{L}_{NO}$), and diffusion ($\mathcal{L}_{Diff}$) losses to balance texture realism, structural coherence, and efficiency.

### Architecture

| Component | Role |
|---|---|
| **Latent Autoencoder** | Compresses HR images into a compact latent representation and decodes latent codes back to pixel space |
| **Fourier Neural Operator (FNO)** | Learns a structure-aware prior in the frequency domain that captures global spatial patterns |
| **Implicit Amplification** | Scale-conditioned MLP that modulates latent features for continuous-scale SR |
| **Latent Diffusion Model** | Iteratively denoises a latent vector, guided by the FNO prior and scale conditioning |

### Training Strategy

Training proceeds in **three sequential stages** (each freezing earlier components):

1. **Stage 1 — Autoencoder** (20 epochs): Minimize $\mathcal{L}_{AE} = \|I_{HR} - \hat{I}_{HR}\|_1$ (L1 reconstruction loss)
2. **Stage 2 — Neural Operator** (15 epochs): Minimize $\mathcal{L}_{NO} = \|z_{HR} - \hat{z}_{NO}\|_2^2$ (MSE in latent space)
3. **Stage 3 — Diffusion UNet** (30 epochs): Minimize $\mathcal{L}_{Diff} = \mathbb{E}[\|\epsilon - \epsilon_\theta(x_t, t, c)\|_2^2]$ (noise prediction MSE)

### Overall Objective Function

$$\mathcal{L}_{total} = \mathcal{L}_{AE} + \lambda_1 \mathcal{L}_{NO} + \lambda_2 \mathcal{L}_{Diff}$$

where $\lambda_1$ and $\lambda_2$ control the contribution of latent mapping and diffusion refinement respectively.

### Evaluation Metrics

| Metric | Formula | Interpretation |
|---|---|---|
| **PSNR** (dB) | $10 \log_{10}\left(\frac{MAX_I^2}{MSE}\right)$ | Pixel-level fidelity. Higher = better. |
| **SSIM** | $\frac{(2\mu_x\mu_y + c_1)(2\sigma_{xy} + c_2)}{(\mu_x^2 + \mu_y^2 + c_1)(\sigma_x^2 + \sigma_y^2 + c_2)}$ | Structural similarity. Higher = better (max 1.0). |
| **LPIPS** | $\sum_l w_l \|\phi_l(I_{HR}) - \phi_l(\hat{I}_{HR})\|_2^2$ | Perceptual distance via deep features. Lower = better. |

### Comparison with State-of-the-Art (Paper Table I)

| Model | PSNR (dB) | SSIM | LPIPS |
|-------|-----------|------|-------|
| LIIF | 27.63 | 0.817 | 0.210 |
| SADN | 28.09 | 0.824 | 0.198 |
| IDM | 28.22 | 0.828 | 0.192 |
| EDiffSR | 28.47 | 0.831 | 0.185 |
| SR3 | 28.95 | 0.842 | 0.173 |
| SPSR | 29.02 | 0.849 | 0.168 |
| HAT-L | 29.21 | 0.852 | 0.165 |
| E²DiffSR (2024) | 28.177 | — | 0.180 |
| NeurOp-Diff (2025) | 27.74 | 0.6509 | — |
| **HNDSR (Ours)** | **29.40** | **0.870** | **0.160** |

### Databricks & MLflow Integration

This notebook runs end-to-end on a **Databricks GPU cluster**. MLflow is used to:
- **Track hyperparameters** (`mlflow.log_params`)
- **Log per-epoch metrics** (`mlflow.log_metrics` with `step=epoch`)
- **Store model checkpoints** as artifacts
- **Register the final model** in the MLflow Model Registry for versioned deployment

---

## Parameters & Hyperparameters Reference

All key parameters used throughout this notebook in one place for easy tuning.

### Model Architecture

| Parameter | Value | Description |
|---|---|---|
| `ae_latent_dim` | 128 | Number of channels in the autoencoder's latent space |
| `ae_downsample_ratio` | 8 | Spatial downsampling factor of the encoder (3 stride-2 convolutions: $2^3 = 8$) |
| `num_res_blocks` | 4 | Residual blocks in both encoder and decoder |
| `no_width` | 32 | Width (hidden channels) of the Fourier Neural Operator |
| `no_modes` | 8 | Number of Fourier modes retained in each spatial dimension |
| `diffusion_channels` | 64 | Base channel count of the Diffusion UNet |
| `num_timesteps` | 1000 | Total diffusion steps ($T$) for the DDPM noise schedule |
| `context_dim` | 128 | Dimension of the cross-attention context vector (same as `ae_latent_dim`) |

### Training Hyperparameters

| Parameter | Stage 1 (AE) | Stage 2 (FNO) | Stage 3 (Diffusion) |
|---|---|---|---|
| Epochs | 20 | 15 | 30 |
| Learning rate | $1 \times 10^{-4}$ | $1 \times 10^{-4}$ | $1 \times 10^{-4}$ |
| Optimizer | AdamW | AdamW | AdamW |
| Weight decay | $1 \times 10^{-4}$ | $1 \times 10^{-4}$ | $1 \times 10^{-4}$ |
| LR scheduler | CosineAnnealingLR | CosineAnnealingLR | CosineAnnealingLR |
| Loss function | L1 (MAE) | MSE | MSE (noise prediction) |
| Batch size | 2 | 2 | 2 |
| Patch size | 64 × 64 | 64 × 64 | 64 × 64 |

### Diffusion Schedule

| Parameter | Value |
|---|---|
| $\beta_\text{start}$ | 0.0001 |
| $\beta_\text{end}$ | 0.02 |
| Schedule type | Linear |
| Inference steps | 50 (DDIM sampling) |

### Other

| Parameter | Value |
|---|---|
| Random seed | 42 |
| Train/Val split | 90% / 10% |
| Scale factor | 4× |
| Image normalization | $[-1, 1]$ (mean=0.5, std=0.5 per channel) |

---

## Cell 0: Databricks Environment & MLflow Setup

This cell configures the **MLflow experiment** and sets up **persistent checkpoint paths on DBFS** (Databricks File System). Key actions:

- `mlflow.set_experiment(...)` — creates or selects the MLflow experiment where all runs will be logged.
- `mlflow.pytorch.autolog(...)` — enables lightweight automatic logging of PyTorch training metrics.
- All checkpoint paths point to `/dbfs/FileStore/...` which persists across cluster restarts.

In [ ]:
# Cell 0: Databricks Environment & MLflow Setup
import os
from pathlib import Path

# ======================================================================
# MLflow is pre-installed on Databricks. If running locally, uncomment:
# %pip install mlflow -q
# ======================================================================
import mlflow
import mlflow.pytorch

# ---------- Experiment Configuration ----------
EXPERIMENT_NAME = "/Users/your-email@domain.com/HNDSR_Satellite_SR"  # <-- UPDATE with your Databricks user path
REGISTERED_MODEL_NAME = "HNDSR_Satellite_SuperResolution"

mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.pytorch.autolog(log_every_n_epoch=1, log_models=False)  # lightweight autolog

print(f"MLflow Tracking URI : {mlflow.get_tracking_uri()}")
print(f"MLflow Experiment   : {EXPERIMENT_NAME}")

# ---------- Persistent Save Location (DBFS) ----------
CHECKPOINT_DIR = "/dbfs/FileStore/hndsr_checkpoints"
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)

AUTOENCODER_PATH     = os.path.join(CHECKPOINT_DIR, 'autoencoder_best.pth')
NEURAL_OPERATOR_PATH = os.path.join(CHECKPOINT_DIR, 'neural_operator_best.pth')
DIFFUSION_PATH       = os.path.join(CHECKPOINT_DIR, 'diffusion_best.pth')
COMPLETE_MODEL_PATH  = os.path.join(CHECKPOINT_DIR, 'hndsr_complete.pth')
EVAL_RESULTS_DIR     = os.path.join(CHECKPOINT_DIR, 'evaluation_results')

print("\n" + "="*70)
print("Checkpoint paths (DBFS - persistent across clusters):")
print(f"   Autoencoder      : {AUTOENCODER_PATH}")
print(f"   Neural Operator  : {NEURAL_OPERATOR_PATH}")
print(f"   Diffusion        : {DIFFUSION_PATH}")
print(f"   Complete Model   : {COMPLETE_MODEL_PATH}")
print(f"   Eval Results     : {EVAL_RESULTS_DIR}")
print("="*70)

## Cell 1: Install Dependencies and Imports

Installs required Python packages and imports all modules used throughout the notebook.

**Line-by-line breakdown:**

- `%pip install torch torchvision lpips timm einops -q` — installs PyTorch, torchvision (image utilities), LPIPS (perceptual loss metric), timm (image model library), and einops (tensor manipulation). The `-q` flag suppresses verbose output.
- `torch`, `torch.nn`, `torch.nn.functional` — core PyTorch modules for tensor operations, neural network layers, and functional ops (activation functions, interpolation, etc.).
- `torch.utils.data.Dataset / DataLoader` — for defining custom datasets and batched data loading with shuffling and parallelism.
- `torch.amp.autocast / GradScaler` — mixed-precision training utilities (imported but not used in this notebook since cuFFT requires float32).
- `torchvision.transforms` — image preprocessing transforms (ToTensor, Normalize, CenterCrop, etc.).
- `torchvision.utils.save_image` — saves tensors as PNG images.
- `numpy`, `math`, `pathlib.Path` — standard numerical, mathematical, and filesystem utilities.
- `PIL.Image` — opens image files from disk.
- `tqdm` — displays progress bars during training loops.
- `lpips` — computes Learned Perceptual Image Patch Similarity comparing SR and HR images.
- `skimage.metrics.peak_signal_noise_ratio / structural_similarity` — computes PSNR and SSIM metrics.
- `matplotlib.pyplot` — used for plotting loss curves and image comparisons.
- `set_seed(42)` — fixes all random seeds (Python, NumPy, PyTorch, CUDA) for reproducibility. `torch.backends.cudnn.deterministic = True` ensures deterministic convolution algorithms.
- `torch.cuda.empty_cache()` — frees unused GPU memory at startup.
- `PYTORCH_CUDA_ALLOC_CONF = 'expandable_segments:True'` — allows the CUDA memory allocator to grow dynamically, reducing out-of-memory errors.

In [ ]:
# Cell 1: Install Dependencies and Imports
%pip install torch torchvision lpips timm einops -q

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import torchvision.transforms as transforms
from torchvision.utils import save_image

import numpy as np
import math
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import lpips
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt
import os
import random
import gc

# Set random seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

# Memory optimization
torch.cuda.empty_cache()
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.2f} GB")

## Dataset Setup (Databricks DBFS)

Sets HR (high-resolution) and LR (low-resolution) image directory paths pointing to Databricks File Store (DBFS), and verifies that the dataset exists.

**Line-by-line breakdown:**

- `HR_DIR = Path("/dbfs/FileStore/datasets/satellite_sr/hr")` — path to high-resolution ground truth images stored in DBFS.
- `LR_DIR = Path("/dbfs/FileStore/datasets/satellite_sr/lr")` — path to corresponding 4× downsampled low-resolution inputs.
- `check_dataset_paths()` — prints whether each directory exists and lists a few sample files, helping confirm the dataset was uploaded correctly. If the check fails, you need to upload the Kaggle `4x-satellite-image-super-resolution` dataset to these DBFS paths via the Databricks UI or CLI.

In [ ]:
# Cell 1.1: Set Dataset Paths on Databricks
#
# Option A  – Data already uploaded to DBFS:
#   HR_DIR = "/dbfs/FileStore/datasets/satellite_sr/HR"
#   LR_DIR = "/dbfs/FileStore/datasets/satellite_sr/LR"
#
# Option B  – Copy from cloud storage first:
#   dbutils.fs.cp("s3://my-bucket/satellite_sr/", "dbfs:/FileStore/datasets/satellite_sr/", recurse=True)
#
# Option C  – Use Databricks Datasets / Unity Catalog Volumes:
#   HR_DIR = "/Volumes/catalog/schema/volume/HR"
#   LR_DIR = "/Volumes/catalog/schema/volume/LR"

# =====================================================================
# SET YOUR PATHS HERE
# =====================================================================
HR_DIR = "/dbfs/FileStore/datasets/satellite_sr/HR"
LR_DIR = "/dbfs/FileStore/datasets/satellite_sr/LR"
# =====================================================================

from pathlib import Path

def check_dataset_paths(hr_dir, lr_dir):
    """Check if dataset paths exist and list available files"""
    print("\n" + "="*60)
    print("DATASET PATH VERIFICATION")
    print("="*60)

    hr_path = Path(hr_dir)
    lr_path = Path(lr_dir)

    print(f"\nHR Directory: {hr_dir}")
    print(f"   Exists: {hr_path.exists()}")

    if hr_path.exists():
        hr_files = list(hr_path.glob('*'))
        print(f"   Total files: {len(hr_files)}")
        if hr_files:
            print(f"   First 5 files: {[f.name for f in hr_files[:5]]}")
            extensions = set([f.suffix.lower() for f in hr_files if f.is_file()])
            print(f"   File extensions: {extensions}")
    else:
        print("   Directory does not exist!")

    print(f"\nLR Directory: {lr_dir}")
    print(f"   Exists: {lr_path.exists()}")

    if lr_path.exists():
        lr_files = list(lr_path.glob('*'))
        print(f"   Total files: {len(lr_files)}")
        if lr_files:
            print(f"   First 5 files: {[f.name for f in lr_files[:5]]}")
            extensions = set([f.suffix.lower() for f in lr_files if f.is_file()])
            print(f"   File extensions: {extensions}")
    else:
        print("   Directory does not exist!")

    print("\n" + "="*60)

check_dataset_paths(HR_DIR, LR_DIR)

## Cell 2: SatelliteDataset Class

Custom PyTorch `Dataset` for loading paired HR/LR satellite image patches.

**Line-by-line breakdown:**

- `class SatelliteDataset(Dataset)` — inherits from `torch.utils.data.Dataset` so it integrates with `DataLoader`.
- `__init__` — finds all `.png` HR images, sorts them alphabetically, and stores the transform pipeline. Prints the number of images found.
- `__len__` — returns total number of image pairs (required by DataLoader to compute batch count).
- `__getitem__` — loads one HR/LR pair by index:
  - Opens both images with `PIL.Image.open().convert('RGB')`.
  - Applies `self.transform` (resize + center crop + normalize) to both.
  - Returns a dict `{'hr': hr_tensor, 'lr': lr_tensor}`.
- **Transform pipeline** (defined outside the class):
  - `Resize(72)` — resizes the shorter edge to 72 px (preserves aspect ratio).
  - `CenterCrop(64)` — crops to a fixed 64×64 patch (the `patch_size` hyperparameter).
  - `ToTensor()` — converts PIL image (H,W,C uint8 0–255) to PyTorch tensor (C,H,W float 0–1).
  - `Normalize([0.5]*3, [0.5]*3)` — rescales pixel values from [0,1] to [-1,1] for stable training.
- **DataLoader** — wraps the dataset with `batch_size=2`, `shuffle=True`, `num_workers=2` for parallel loading, and `pin_memory=True` for faster CPU→GPU transfers.

In [ ]:
# Cell 3: Dataset Class (Memory Optimized - Power of 2 patch size)
class SatelliteDataset(Dataset):
    """Dataset for satellite image super-resolution"""
    def __init__(self, hr_dir, lr_dir, patch_size=64, training=True):
        self.hr_dir = Path(hr_dir)
        self.lr_dir = Path(lr_dir)
        self.patch_size = patch_size
        self.training = training

        # Try multiple image extensions
        self.hr_images = []
        self.lr_images = []

        for ext in ['*.png', '*.jpg', '*.jpeg', '*.tif', '*.tiff', '*.PNG', '*.JPG', '*.JPEG', '*.TIF', '*.TIFF']:
            self.hr_images.extend(list(self.hr_dir.glob(ext)))
            self.lr_images.extend(list(self.lr_dir.glob(ext)))

        self.hr_images = sorted(self.hr_images)
        self.lr_images = sorted(self.lr_images)

        if len(self.hr_images) == 0 or len(self.lr_images) == 0:
            raise ValueError(
                f"No images found!\n"
                f"HR Directory: {hr_dir} ({len(self.hr_images)} images)\n"
                f"LR Directory: {lr_dir} ({len(self.lr_images)} images)\n"
                f"Please check if the paths are correct."
            )

        # Match images by filename
        hr_names = {img.stem: img for img in self.hr_images}
        lr_names = {img.stem: img for img in self.lr_images}

        # Find common images
        common_names = set(hr_names.keys()) & set(lr_names.keys())

        if len(common_names) == 0:
            print("Warning: No matching filenames found between HR and LR directories!")
            print(f"   HR samples: {list(hr_names.keys())[:5]}")
            print(f"   LR samples: {list(lr_names.keys())[:5]}")
            # Use all images if no matches (assume same order)
            min_len = min(len(self.hr_images), len(self.lr_images))
            self.hr_images = self.hr_images[:min_len]
            self.lr_images = self.lr_images[:min_len]
        else:
            # Use only matching images
            self.hr_images = [hr_names[name] for name in sorted(common_names)]
            self.lr_images = [lr_names[name] for name in sorted(common_names)]

        print(f"Found {len(self.hr_images)} image pairs")
        if len(self.hr_images) > 0:
            print(f"Sample HR: {self.hr_images[0].name}")
            print(f"Sample LR: {self.lr_images[0].name}")

        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

    def __len__(self):
        return len(self.hr_images)

    def __getitem__(self, idx):
        hr_img = Image.open(self.hr_images[idx]).convert('RGB')
        lr_img = Image.open(self.lr_images[idx]).convert('RGB')

        if self.training:
            hr_w, hr_h = hr_img.size
            lr_w, lr_h = lr_img.size
            scale = hr_w // lr_w

            lr_crop_size = self.patch_size // scale
            if lr_w > lr_crop_size and lr_h > lr_crop_size:
                x = random.randint(0, lr_w - lr_crop_size)
                y = random.randint(0, lr_h - lr_crop_size)

                lr_img = lr_img.crop((x, y, x + lr_crop_size, y + lr_crop_size))
                hr_img = hr_img.crop((x * scale, y * scale,
                                     (x + lr_crop_size) * scale,
                                     (y + lr_crop_size) * scale))

            if random.random() > 0.5:
                lr_img = lr_img.transpose(Image.FLIP_LEFT_RIGHT)
                hr_img = hr_img.transpose(Image.FLIP_LEFT_RIGHT)
        else:
            hr_img = transforms.CenterCrop(self.patch_size)(hr_img)
            lr_img = transforms.CenterCrop(self.patch_size // 4)(lr_img)

        lr_tensor = self.transform(lr_img)
        hr_tensor = self.transform(hr_img)

        return {'lr': lr_tensor, 'hr': hr_tensor, 'scale': 4}

## Cell 3: Latent Autoencoder (Stage 1 Architecture)

The first component of the HNDSR pipeline — **Stage 1: Autoencoder Residual Learning** (Paper §III.A).

Compresses HR images into a compact latent space and reconstructs them. Acts as a feature extractor providing latent representations $z_{HR} = E_\theta(I_{HR})$ for downstream stages.

**Mathematical formulation (from paper):**

$$\hat{I}_{HR} = D_\phi(E_\theta(I_{HR}))$$

$$\mathcal{L}_{AE} = \|I_{HR} - \hat{I}_{HR}\|_1$$

where $E_\theta$ is the encoder, $D_\phi$ is the decoder, and L1 loss enforces sharp edge preservation (preferred over L2 which produces blurring).

**Architecture overview:**

| Component | Details |
|-----------|---------|
| Encoder | Conv2d(3→64) → 3× ResidualBlock(64) → Conv2d(64→128, stride=2) ×3 → Conv2d(128→`ae_latent_dim`) |
| Decoder | ConvTranspose2d(`ae_latent_dim`→128) → 3× [Upsample(2×) + Conv2d(128→128)] → Conv2d(128→64) → ResidualBlock(64) → Conv2d(64→3) + Tanh |
| Latent shape | $(B, 128, H/8, W/8)$ — spatial dimensions reduced by $2^3 = 8\times$ |

**Line-by-line breakdown:**

- `class ResidualBlock(nn.Module)` — a convolutional block with a skip connection:
  - Two Conv2d(ch, ch, 3, pad=1) + BatchNorm + ReLU layers.
  - `forward()` adds input to output (`x + self.block(x)`), enabling gradient flow and avoiding vanishing gradients.
- `class LatentAutoencoder(nn.Module)`:
  - **Encoder** — progressively downsamples spatial dimensions by 8× (3 stride-2 convolutions). Each Conv2d is followed by BatchNorm and LeakyReLU(0.2). Final conv projects to `latent_dim` channels.
  - **Decoder** — mirrors the encoder using nearest-neighbor upsampling + conv (avoids checkerboard artifacts from ConvTranspose2d). Final `Tanh()` outputs values in [-1,1] matching the input normalization.
  - `encode(x)` / `decode(z)` — separate methods so other components can access the latent space directly.
  - `forward(x)` — calls `encode` then `decode`, returning both the reconstruction and the latent tensor.

In [ ]:
# Cell 4: Latent Autoencoder (Reduced Size)
class ResidualBlock(nn.Module):
    """Residual block for autoencoder"""
    def __init__(self, channels, use_bn=False):
        super().__init__()
        layers = [
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1)
        ]
        if use_bn:
            layers.insert(1, nn.BatchNorm2d(channels))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return x + self.block(x)


class LatentAutoencoder(nn.Module):
    """Autoencoder for learning latent representation"""
    def __init__(self, in_channels=3, latent_dim=64, num_res_blocks=4, downsample_ratio=8):
        super().__init__()
        self.latent_dim = latent_dim
        self.downsample_ratio = downsample_ratio

        num_downs = int(math.log2(downsample_ratio))

        # Encoder
        encoder_layers = [nn.Conv2d(in_channels, latent_dim, 3, padding=1)]
        channels = latent_dim
        for i in range(num_downs):
            out_channels = min(channels * 2, 128)
            encoder_layers.extend([
                nn.Conv2d(channels, out_channels, 4, stride=2, padding=1),
                nn.ReLU(inplace=True)
            ])
            channels = out_channels

        for _ in range(num_res_blocks):
            encoder_layers.append(ResidualBlock(channels))

        self.encoder = nn.Sequential(*encoder_layers)

        # Decoder
        decoder_layers = []
        for i in range(num_res_blocks):
            decoder_layers.append(ResidualBlock(channels))

        for i in range(num_downs):
            out_channels = channels // 2
            decoder_layers.extend([
                nn.ConvTranspose2d(channels, out_channels, 4, stride=2, padding=1),
                nn.ReLU(inplace=True)
            ])
            channels = out_channels

        decoder_layers.append(nn.Conv2d(channels, in_channels, 3, padding=1))
        decoder_layers.append(nn.Tanh())

        self.decoder = nn.Sequential(*decoder_layers)

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        z = self.encode(x)
        recon = self.decode(z)
        return recon, z

## Cell 4: Fourier Neural Operator — FNO (Stage 2 Architecture)

The second component — **Stage 2: Neural Operator-Based Latent Mapping** (Paper §III.B).

Inspired by the Fourier Neural Operator framework [Li et al., 2020], this module learns a continuous-scale mapping from LR to HR latent features. Neural operators map between **infinite-dimensional function spaces** rather than discrete pixel grids, enabling dynamic adaptation to arbitrary resolutions.

**Mathematical formulation (from paper):**

$$\hat{z}_{NO} = N_\psi(I_{LR}, s)$$

$$\mathcal{L}_{NO} = \|z_{HR} - \hat{z}_{NO}\|_2^2$$

where $N_\psi$ captures long-range dependencies using frequency-domain transformations, $s$ is the scale factor, and $z_{HR}$ is the target latent from Stage 1.

**Key idea:** Convolutions in the spatial domain equal pointwise multiplications in the Fourier domain. The FNO learns a filter in frequency space, capturing long-range dependencies efficiently in $O(N \log N)$ time. This acts as a **deterministic bridge** that projects low-resolution representations into a learned high-dimensional latent space.

**Line-by-line breakdown:**

- `class SpectralConv2d(nn.Module)` — the core spectral convolution layer:
  - `self.weights1 / weights2` — learnable complex-valued weight tensors of shape `(in_ch, out_ch, modes, modes)`, initialized with Xavier uniform.
  - `compl_mul2d()` — performs Einstein summation (`bixy,ioxy->boxy`) to multiply input frequency modes by learned weights.
  - `forward()` — applies `torch.fft.rfft2` (real 2D FFT) to input, multiplies low-frequency modes (up to `self.modes`) with learned weights, pads remaining modes with zeros, applies `torch.fft.irfft2` (inverse FFT) to return to spatial domain, and adds a residual 1×1 convolution path.
  - **Float32 enforcement** — cuFFT on NVIDIA GPUs doesn't support half-precision, so input is cast to `float32` before FFT and back afterward.
- `class NeuralOperator(nn.Module)`:
  - `self.fc0` — lifts `ae_latent_dim + 1` input channels (latent + scale map) to `width` (32) channels.
  - 4× `SpectralConv2d` layers — each followed by the residual 1×1 conv and GELU activation.
  - `self.fc1 / fc2` — projects back from `width` to `ae_latent_dim` output channels.
  - `scale_map` — a spatial tensor filled with `scale_factor / 4.0`, concatenated as an extra input channel to enable continuous-scale conditioning.
  - `forward()` iterates through spectral layers with `F.gelu` activations, then applies the output projections.

In [ ]:
# Cell 5: Neural Operator (Fixed for cuFFT + non-power-of-2 sizes)
class SpectralConv2d(nn.Module):
    """Spectral convolution for Fourier Neural Operator - Fixed for mixed precision and cuFFT"""
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.modes2 = modes2

        self.scale = 1 / (in_channels * out_channels)
        self.weights1 = nn.Parameter(
            self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, 2)
        )
        self.weights2 = nn.Parameter(
            self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, 2)
        )

    def forward(self, x):
        batchsize = x.shape[0]

        # CRITICAL FIX: Convert to float32 for FFT to avoid cuFFT limitations
        x_dtype = x.dtype
        x = x.float()

        # Compute FFT in float32
        x_ft = torch.fft.rfft2(x)

        # Multiply relevant Fourier modes
        out_ft = torch.zeros(batchsize, self.out_channels,
                            x.size(-2), x.size(-1)//2 + 1,
                            dtype=torch.cfloat, device=x.device)

        modes1 = min(self.modes1, x.size(-2))
        modes2 = min(self.modes2, x.size(-1)//2 + 1)

        if modes1 > 0 and modes2 > 0:
            out_ft[:, :, :modes1, :modes2] = \
                self._compl_mul2d(x_ft[:, :, :modes1, :modes2],
                                 torch.view_as_complex(self.weights1[:, :, :modes1, :modes2]))

            out_ft[:, :, -modes1:, :modes2] = \
                self._compl_mul2d(x_ft[:, :, -modes1:, :modes2],
                                 torch.view_as_complex(self.weights2[:, :, :modes1, :modes2]))

        x_out = torch.fft.irfft2(out_ft, s=(x.size(-2), x.size(-1)))

        if x_dtype != torch.float32:
            x_out = x_out.to(x_dtype)

        return x_out

    def _compl_mul2d(self, input, weights):
        return torch.einsum("bixy,ioxy->boxy", input, weights)


class NeuralOperator(nn.Module):
    """Neural Operator for structure-aware prior"""
    def __init__(self, in_channels=3, out_channels=128, modes=8, width=32):
        super().__init__()
        self.modes = modes
        self.width = width

        self.fc0 = nn.Conv2d(in_channels + 1, width, 1)

        self.conv0 = SpectralConv2d(width, width, modes, modes)
        self.conv1 = SpectralConv2d(width, width, modes, modes)
        self.conv2 = SpectralConv2d(width, width, modes, modes)
        self.conv3 = SpectralConv2d(width, width, modes, modes)

        self.w0 = nn.Conv2d(width, width, 1)
        self.w1 = nn.Conv2d(width, width, 1)
        self.w2 = nn.Conv2d(width, width, 1)
        self.w3 = nn.Conv2d(width, width, 1)

        self.fc1 = nn.Conv2d(width, 64, 1)
        self.fc2 = nn.Conv2d(64, out_channels, 1)

    def forward(self, x, scale_factor):
        b, c, h, w = x.shape
        scale_map = torch.ones(b, 1, h, w, device=x.device) * (scale_factor / 4.0)
        x = torch.cat([x, scale_map], dim=1)

        x = self.fc0(x)

        x1 = self.conv0(x)
        x2 = self.w0(x)
        x = F.gelu(x1 + x2)

        x1 = self.conv1(x)
        x2 = self.w1(x)
        x = F.gelu(x1 + x2)

        x1 = self.conv2(x)
        x2 = self.w2(x)
        x = F.gelu(x1 + x2)

        x1 = self.conv3(x)
        x2 = self.w3(x)
        x = F.gelu(x1 + x2)

        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)

        return x

## Cell 5: Implicit Amplification Module

A scale-conditioned MLP that modulates latent features for continuous-scale super-resolution, inspired by E²DiffSR's implicit amplification concept [Wu et al., 2024].

**Role in HNDSR:** While the neural operator provides a resolution-agnostic latent mapping, this module adds **scale-specific modulation** — it predicts per-channel gain factors conditioned on the target scale factor $s$, enabling the same model to handle arbitrary upscaling (e.g., 2×, 3.5×, 4×, 6×) without retraining.

**Line-by-line breakdown:**

- `class ImplicitAmplification(nn.Module)`:
  - `self.mlp` — a 3-layer MLP: `Linear(1 → 256) → ReLU → Linear(256 → 256) → ReLU → Linear(256 → ae_latent_dim) → Sigmoid`.
  - The input is a **scalar scale factor** (e.g., 4.0), not the latent itself.
  - `Sigmoid()` output constrains gains to [0, 1], so the final scaling is `latent × (1 + gains)` — a multiplicative residual in range [1, 2].
  - `forward(latent, scale_factor)`:
    - Creates a `(B, 1)` scale input tensor.
    - MLP produces `(B, ae_latent_dim)` gain vector.
    - Reshaped to `(B, C, 1, 1)` and broadcast-multiplied with the latent.
  - This enables continuous-scale SR — a core contribution of HNDSR over fixed-scale methods like SR3.

In [ ]:
# Cell 6: Implicit Amplification Module
class ImplicitAmplification(nn.Module):
    """MLP that predicts channel-wise gains for high-frequency enhancement"""
    def __init__(self, latent_dim=128, hidden_dim=256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, latent_dim),
            nn.Sigmoid()
        )

    def forward(self, latent, scale_factor):
        b, c, h, w = latent.shape

        if isinstance(scale_factor, (int, float)):
            scale_input = torch.full((b, 1), scale_factor, device=latent.device, dtype=torch.float32)
        else:
            scale_input = scale_factor.view(b, 1).float()

        gains = self.mlp(scale_input)
        gains = gains.view(b, c, 1, 1)

        return latent * (1 + gains)

## Cell 6: Diffusion UNet — Attention & Building Blocks (Stage 3 Components)

Defines the building blocks used by the Diffusion UNet — part of **Stage 3: Diffusion-Based Probabilistic Refinement** (Paper §III.C).

The diffusion model employs a DDPM [Ho et al., 2020] to introduce stochastic refinement and generate fine textures typically lost in deterministic reconstruction. The diffusion process progressively corrupts a latent variable $x_0$ into a noisy sample $x_t$ via:

$$q(x_t | x_{t-1}) = \mathcal{N}\left(\sqrt{1 - \beta_t}\, x_{t-1},\; \beta_t\, \mathbf{I}\right)$$

The reverse process is modelled by a U-Net conditioned on the neural operator's context $c$:

$$\epsilon_\theta(x_t, t, c) \approx \epsilon, \quad \mathcal{L}_{Diff} = \mathbb{E}\left[\|\epsilon - \epsilon_\theta(x_t, t, c)\|_2^2\right]$$

This probabilistic denoising refines texture, illumination, and fine-grained features while maintaining global consistency.

**Line-by-line breakdown:**

- `class SinusoidalPositionEmbeddings(nn.Module)`:
  - Encodes the diffusion timestep $t$ into a vector using sinusoidal functions (same idea as Transformer position encodings). Creates a `dim`-dimensional embedding using interleaved sine/cosine at exponentially increasing frequencies.
  - This lets the UNet distinguish between noise levels at different timesteps.

- `class AttentionBlock(nn.Module)`:
  - Single-head self-attention for spatial feature maps.
  - Reshapes $(B,C,H,W) \to (B, H \times W, C)$, applies `nn.MultiheadAttention`, and reshapes back.
  - Adds a residual connection: `output = x + attention(norm(x))`.
  - Enables the network to capture long-range spatial dependencies within feature maps.

- `class CrossAttentionBlock(nn.Module)`:
  - Cross-attention between noisy image features and **neural operator conditioning features** $c$.
  - Query comes from input $x$, Key/Value come from context $c$ (the FNO's refined latent).
  - Both are layer-normalized before attention. Includes residual connection.
  - This is how the diffusion model is **conditioned on the neural operator prior**.

- `class ResidualBlockWithTime(nn.Module)`:
  - A ResidualBlock conditioned on the timestep embedding.
  - `forward(x, t)`: first conv+norm+ReLU → add projected timestep embedding to feature map → second conv+norm+ReLU → add skip connection.
  - The `time_mlp` projects the timestep embedding to match the channel dimension, broadcasting across spatial dims.

In [ ]:
# Cell 7: Diffusion UNet Components
class SinusoidalPositionEmbeddings(nn.Module):
    """Sinusoidal time embeddings for diffusion"""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings


class AttentionBlock(nn.Module):
    """Self-attention block for UNet"""
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        self.norm = nn.GroupNorm(min(8, channels), channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.proj = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        b, c, h, w = x.shape
        residual = x
        x = self.norm(x)

        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=1)

        q = q.view(b, c, h * w).transpose(1, 2)
        k = k.view(b, c, h * w).transpose(1, 2)
        v = v.view(b, c, h * w).transpose(1, 2)

        scale = c ** -0.5
        attn = torch.softmax(torch.bmm(q, k.transpose(1, 2)) * scale, dim=-1)
        out = torch.bmm(attn, v)

        out = out.transpose(1, 2).view(b, c, h, w)
        out = self.proj(out)

        return out + residual


class CrossAttentionBlock(nn.Module):
    """Cross-attention for conditioning"""
    def __init__(self, channels, context_dim):
        super().__init__()
        self.channels = channels
        self.norm = nn.GroupNorm(min(8, channels), channels)
        self.q = nn.Conv2d(channels, channels, 1)
        self.kv = nn.Linear(context_dim, channels * 2)
        self.proj = nn.Conv2d(channels, channels, 1)

    def forward(self, x, context):
        b, c, h, w = x.shape
        residual = x
        x = self.norm(x)

        q = self.q(x).view(b, c, h * w).transpose(1, 2)

        kv = self.kv(context)
        k, v = kv.chunk(2, dim=1)
        k = k.unsqueeze(1)
        v = v.unsqueeze(1)

        scale = c ** -0.5
        attn = torch.softmax(torch.bmm(q, k.transpose(1, 2)) * scale, dim=-1)
        out = torch.bmm(attn, v)

        out = out.transpose(1, 2).view(b, c, h, w)
        out = self.proj(out)

        return out + residual


class ResidualBlockWithTime(nn.Module):
    """Residual block with time embedding"""
    def __init__(self, in_channels, out_channels, time_embed_dim):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels

        self.norm1 = nn.GroupNorm(min(8, in_channels), in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)

        self.time_emb = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_embed_dim, out_channels)
        )

        self.norm2 = nn.GroupNorm(min(8, out_channels), out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)

        if in_channels != out_channels:
            self.shortcut = nn.Conv2d(in_channels, out_channels, 1)
        else:
            self.shortcut = nn.Identity()

    def forward(self, x, t_emb):
        h = self.norm1(x)
        h = F.silu(h)
        h = self.conv1(h)

        h = h + self.time_emb(t_emb)[:, :, None, None]

        h = self.norm2(h)
        h = F.silu(h)
        h = self.conv2(h)

        return h + self.shortcut(x)

## Cell 7: DiffusionUNet — Full U-Net Architecture (Stage 3 Network)

The denoising network for Stage 3. A U-Net conditioned on timestep $t$ and context $c$ (the neural operator's refined latent), used to iteratively denoise the super-resolution output.

This implements the reverse process model $\epsilon_\theta(x_t, t, c)$ from Eq. III.C of the paper. The conditioning on $c$ is what makes this a **neural operator–guided diffusion model**, distinguishing HNDSR from standard DDPM/SR3 which condition only on the LR image.

**Architecture (channels = 64):**

```
Input (ae_latent_dim=128)
  ↓ init_conv → (64)
  ↓ down1: ResBlock+Time(64→64) + Attention + CrossAttn(c) → skip1
  ↓ pool(2×)
  ↓ down2: ResBlock+Time(64→128) + Attention + CrossAttn(c) → skip2
  ↓ pool(2×)
  ↓ mid: ResBlock+Time(128→128) + Attention + CrossAttn(c) + ResBlock
  ↓ up2: concat(skip2) → ResBlock+Time(256→128) + Attention + CrossAttn(c)
  ↓ upsample(2×)
  ↓ up1: concat(skip1) → ResBlock+Time(192→64) + Attention + CrossAttn(c)
  ↓ upsample(2×)
  ↓ final_conv → (ae_latent_dim=128)
```

**Line-by-line breakdown:**

- `self.time_mlp` — projects sinusoidal timestep embedding through `Linear → GELU → Linear` to produce a $4 \times \text{channels}$ vector used by all `ResidualBlockWithTime` layers.
- `self.context_proj` — 1×1 conv that maps the neural operator context $c$ to match the UNet's internal channel width. This is the key bridge between Stage 2 and Stage 3.
- **Down path** (`down1`, `down2`) — each level: time-conditioned ResBlock → self-attention → cross-attention with context $c$ → max pool.
- **Mid block** — bottleneck with double ResBlock + attention + cross-attention at the lowest resolution.
- **Up path** (`up1`, `up2`) — upsamples with `F.interpolate(scale_factor=2)`, concatenates skip connections from the encoder, then ResBlock + attention + cross-attention.
- **Channel arithmetic** for `up2`: 128 (upsampled) + 128 (skip) = 256 → ResBlock reduces to 128. For `up1`: 128 + 64 (skip) = 192 → ResBlock reduces to 64.
- `final_conv` — `Conv2d(64 → ae_latent_dim)` returns to original latent dimension.

In [ ]:
# Cell 8: Simplified Diffusion UNet (FIXED - Concatenation dimension bug)
class DiffusionUNet(nn.Module):
    """Simplified UNet for latent diffusion - Memory Optimized"""
    def __init__(self, in_channels=128, model_channels=64, out_channels=128, context_dim=128):
        super().__init__()

        self.in_channels = in_channels
        self.model_channels = model_channels

        # Time embedding
        time_embed_dim = model_channels * 4
        self.time_embed = nn.Sequential(
            SinusoidalPositionEmbeddings(model_channels),
            nn.Linear(model_channels, time_embed_dim),
            nn.SiLU(),
            nn.Linear(time_embed_dim, time_embed_dim)
        )

        # Simplified architecture
        self.input_proj = nn.Conv2d(in_channels, model_channels, 3, padding=1)

        # Down
        self.down1 = ResidualBlockWithTime(model_channels, model_channels * 2, time_embed_dim)
        self.down2 = nn.Conv2d(model_channels * 2, model_channels * 2, 3, stride=2, padding=1)

        # Middle with cross-attention
        self.mid1 = ResidualBlockWithTime(model_channels * 2, model_channels * 2, time_embed_dim)
        self.mid_attn = CrossAttentionBlock(model_channels * 2, context_dim)
        self.mid2 = ResidualBlockWithTime(model_channels * 2, model_channels * 2, time_embed_dim)

        # Up
        self.up1 = nn.ConvTranspose2d(model_channels * 2, model_channels * 2, 4, stride=2, padding=1)
        self.up2 = ResidualBlockWithTime(model_channels * 3, model_channels, time_embed_dim)

        # Output
        self.out = nn.Sequential(
            nn.GroupNorm(8, model_channels),
            nn.SiLU(),
            nn.Conv2d(model_channels, out_channels, 3, padding=1)
        )

    def forward(self, x, t, context):
        t_emb = self.time_embed(t)

        h = self.input_proj(x)
        h0 = h

        h = self.down1(h, t_emb)
        h = self.down2(h)

        h = self.mid1(h, t_emb)
        h = self.mid_attn(h, context)
        h = self.mid2(h, t_emb)

        h = self.up1(h)
        h = torch.cat([h, h0], dim=1)
        h = self.up2(h, t_emb)

        return self.out(h)

## Cell 8: DDPM Noise Scheduler

Implements the Denoising Diffusion Probabilistic Model (DDPM) [Ho et al., 2020] noise schedule — controls how noise is added during training (forward process) and removed during inference (reverse process).

**Forward process — closed-form sampling (Paper §III.C):**

Given clean data $x_0$, we can sample any noisy version $x_t$ directly without iterating:

$$x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \epsilon, \quad \epsilon \sim \mathcal{N}(0, \mathbf{I})$$

where $\bar{\alpha}_t = \prod_{i=1}^{t} (1 - \beta_i)$ is the cumulative signal retention.

**Reverse process — DDPM update rule:**

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}}\,\epsilon_\theta(x_t, t, c)\right) + \sigma_t\, z, \quad z \sim \mathcal{N}(0, \mathbf{I})$$

where $\epsilon_\theta$ is the predicted noise from the UNet, $c$ is the neural operator context, and $\sigma_t^2 = \frac{\beta_t (1 - \bar{\alpha}_{t-1})}{1 - \bar{\alpha}_t}$ is the posterior variance.

**Line-by-line breakdown:**

- `class DDPMScheduler`:
  - `__init__` — creates a **linear** beta schedule from $\beta_{start}=0.0001$ to $\beta_{end}=0.02$ over $T=1000$ steps.
  - Precomputes: `alphas`, `alphas_cumprod` ($\bar{\alpha}_t$), `sqrt_alphas_cumprod`, `sqrt_one_minus_alphas_cumprod`, and `posterior_variance`.
  - `add_noise(x, noise, timesteps)` — the **forward process**: implements the closed-form $x_t$ formula above. Handles device placement (indices on CPU for gather, tensors on GPU for math).
  - `ddim_sample(model_output, timestep, sample)` — the **DDIM reverse step** (deterministic sampling, faster than full DDPM). Predicts $x_0$ from $x_t$ and the noise prediction, then jumps directly to $x_{t-1}$.

In [ ]:
# Cell 9: DDPM Scheduler (FIXED - Device handling)
class DDPMScheduler:
    """DDPM noise scheduler with proper device handling"""
    def __init__(self, num_timesteps=1000, beta_start=0.0001, beta_end=0.02):
        self.num_timesteps = num_timesteps

        self.betas = torch.linspace(beta_start, beta_end, num_timesteps)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)

        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)

        self.posterior_variance = (
            self.betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        )

    def add_noise(self, x_start, noise, timesteps):
        timesteps_cpu = timesteps.cpu()

        sqrt_alpha_prod = self.sqrt_alphas_cumprod[timesteps_cpu].to(x_start.device)
        sqrt_one_minus_alpha_prod = self.sqrt_one_minus_alphas_cumprod[timesteps_cpu].to(x_start.device)

        while len(sqrt_alpha_prod.shape) < len(x_start.shape):
            sqrt_alpha_prod = sqrt_alpha_prod.unsqueeze(-1)
            sqrt_one_minus_alpha_prod = sqrt_one_minus_alpha_prod.unsqueeze(-1)

        return sqrt_alpha_prod * x_start + sqrt_one_minus_alpha_prod * noise

    def ddim_sample(self, model_output, timestep, sample):
        if isinstance(timestep, torch.Tensor):
            t = timestep.item() if timestep.numel() == 1 else timestep
        else:
            t = timestep

        alpha_prod_t = self.alphas_cumprod[t].to(sample.device)
        alpha_prod_t_prev = self.alphas_cumprod_prev[t].to(sample.device) if t > 0 else torch.tensor(1.0).to(sample.device)

        beta_prod_t = 1 - alpha_prod_t

        pred_original_sample = (sample - torch.sqrt(beta_prod_t) * model_output) / torch.sqrt(alpha_prod_t)
        pred_sample_direction = torch.sqrt(1 - alpha_prod_t_prev) * model_output
        pred_prev_sample = torch.sqrt(alpha_prod_t_prev) * pred_original_sample + pred_sample_direction

        return pred_prev_sample

## Cell 9: Complete HNDSR Model

Combines all four components into a single `nn.Module` — the full **Hybrid Neural Operator-Diffusion Super-Resolution** pipeline (Paper §III).

**Overall Objective Function (Paper §III.D):**

$$\mathcal{L}_{total} = \mathcal{L}_{AE} + \lambda_1 \mathcal{L}_{NO} + \lambda_2 \mathcal{L}_{Diff}$$

In practice, the three stages are trained **sequentially** (not jointly), so each loss is optimized independently with its own optimizer. This avoids gradient interference between the deterministic (AE, FNO) and stochastic (diffusion) components.

**Neural Operator Prior (Paper §II.D):**

The neural operator learns a continuous mapping $N_\phi : \mathbb{R}^{H \times W} \to \mathcal{F}(\Omega)$, where $\mathcal{F}(\Omega)$ is a function space over spatial coordinates. This prior captures long-range relationships and structural continuity across scales. The latent diffusion backbone $D_\theta$ then denoises in this compact feature domain, conditioned on the neural operator embeddings.

**Line-by-line breakdown:**

- `class HNDSR(nn.Module)`:
  - `__init__` — instantiates: `LatentAutoencoder`, `NeuralOperator`, `ImplicitAmplification`, `DiffusionUNet`, and `DDPMScheduler`.
  - `forward(lr_img, scale_factor=4)` — **training** forward pass:
    1. **Encode** — $z = E_\theta(I_{LR})$, shape $(B, 128, H/8, W/8)$.
    2. **Neural operator** — $z_{refined} = N_\psi(z, s)$, frequency-domain enhancement.
    3. **Implicit amplification** — $z_{amplified} = z_{refined} \times (1 + g(s))$, scale-conditioned modulation.
    4. **Encode HR target** — $z_{target} = E_\theta(I_{HR})$ (for loss computation).
    5. **Add noise** — sample $t \sim \text{Uniform}(0, T)$ and $\epsilon \sim \mathcal{N}(0, I)$, compute $x_t$.
    6. **Predict noise** — $\epsilon_\theta = \text{UNet}(x_t, t, z_{refined})$.
    7. Returns `{'noise_pred', 'noise_target', 'z_amplified', 'z_target', 'z_refined'}`.
  - `super_resolve(lr_img, scale_factor=4, num_steps=50)` — **inference** path:
    1. Encode → neural operator → implicit amplification (deterministic).
    2. DDIM reverse sampling: iteratively denoise from $x_T \sim \mathcal{N}(0, I)$ → $x_0$.
    3. `autoencoder.decode(x_0)` → super-resolved image in $[-1, 1]$.
    4. Returns the SR image tensor.

In [ ]:
# Cell 10: Complete HNDSR Model
class HNDSR(nn.Module):
    """Hybrid Neural Operator-Diffusion Super-Resolution"""
    def __init__(self,
                 ae_latent_dim=128,
                 ae_downsample_ratio=8,
                 no_width=32,
                 no_modes=8,
                 diffusion_channels=64,
                 num_timesteps=1000):
        super().__init__()

        self.autoencoder = LatentAutoencoder(
            in_channels=3,
            latent_dim=ae_latent_dim,
            num_res_blocks=4,
            downsample_ratio=ae_downsample_ratio
        )

        self.neural_operator = NeuralOperator(
            in_channels=3,
            out_channels=ae_latent_dim,
            modes=no_modes,
            width=no_width
        )

        self.implicit_amp = ImplicitAmplification(
            latent_dim=ae_latent_dim,
            hidden_dim=256
        )

        self.diffusion_unet = DiffusionUNet(
            in_channels=ae_latent_dim,
            model_channels=diffusion_channels,
            out_channels=ae_latent_dim,
            context_dim=ae_latent_dim
        )

        self.scheduler = DDPMScheduler(num_timesteps=num_timesteps)
        self.ae_downsample_ratio = ae_downsample_ratio

    def encode_hr(self, hr_img):
        _, z = self.autoencoder(hr_img)
        return z

    def decode_latent(self, z):
        return self.autoencoder.decode(z)

    def get_no_prior(self, lr_img, scale_factor):
        lr_upscaled = F.interpolate(lr_img, scale_factor=scale_factor, mode='bicubic', align_corners=False)
        no_features = self.neural_operator(lr_upscaled, scale_factor)

        latent_size = lr_upscaled.shape[-1] // self.ae_downsample_ratio
        no_prior = F.interpolate(no_features, size=(latent_size, latent_size), mode='bilinear', align_corners=False)

        return no_prior

    @torch.no_grad()
    def super_resolve(self, lr_img, scale_factor=4, num_inference_steps=50):
        device = lr_img.device
        b = lr_img.shape[0]

        no_prior = self.get_no_prior(lr_img, scale_factor)
        no_prior = self.implicit_amp(no_prior, scale_factor)
        context = F.adaptive_avg_pool2d(no_prior, 1).view(b, -1)

        latent_shape = no_prior.shape
        z_t = torch.randn(latent_shape, device=device)

        timesteps = torch.linspace(self.scheduler.num_timesteps - 1, 0, num_inference_steps, dtype=torch.long)
        for t in tqdm(timesteps, desc="Diffusion sampling", leave=False):
            t_batch = torch.full((b,), t, device=device, dtype=torch.long)
            noise_pred = self.diffusion_unet(z_t, t_batch, context)
            z_t = self.scheduler.ddim_sample(noise_pred, t, z_t)

        hr_pred = self.decode_latent(z_t)
        return hr_pred

## Cell 10: Evaluation Metrics — PSNR, SSIM & LPIPS (Paper §III.E)

Defines image quality metrics used to evaluate super-resolution output against ground truth. Three complementary metrics jointly measure numerical accuracy and human perceptual fidelity.

**PSNR — Peak Signal-to-Noise Ratio:**

$$PSNR = 10 \log_{10}\left(\frac{MAX_I^2}{MSE(I_{HR}, \hat{I}_{HR})}\right)$$

Measures pixel-level fidelity. Higher is better. Typical range for 4× SR: 25–30 dB.

**SSIM — Structural Similarity Index:**

$$SSIM(x, y) = \frac{(2\mu_x\mu_y + c_1)(2\sigma_{xy} + c_2)}{(\mu_x^2 + \mu_y^2 + c_1)(\sigma_x^2 + \sigma_y^2 + c_2)}$$

Measures structural similarity (luminance, contrast, structure). Ranges 0 to 1; higher is better.

**LPIPS — Learned Perceptual Image Patch Similarity** (computed in evaluation cell via `lpips` library):

$$LPIPS(x, y) = \sum_l w_l \|\phi_l(x) - \phi_l(y)\|_2^2$$

where $\phi_l(\cdot)$ is the activation at layer $l$ of a pretrained VGG/AlexNet, and $w_l$ are learned weights. Lower LPIPS = higher perceptual quality. This metric captures differences invisible to PSNR/SSIM (e.g., texture realism, sharpness).

**Line-by-line breakdown:**

- `calculate_psnr(sr, hr)`:
  - Converts tensors to numpy, transposes (C,H,W)→(H,W,C), clips to [0,1].
  - Calls `peak_signal_noise_ratio(hr, sr, data_range=1.0)`.

- `calculate_ssim(sr, hr)`:
  - Same preprocessing. Calls `structural_similarity(hr, sr, data_range=1.0, channel_axis=2, win_size=3)`.
  - `win_size=3` (smaller than default 7) accommodates the small 64×64 patch size.

> **Note:** LPIPS is computed in the evaluation function (Cell 14) using the `lpips` library, not in this cell.

In [ ]:
# Cell 11: Loss Functions and Metrics
def calculate_psnr(img1, img2):
    """Calculate PSNR between two images"""
    img1 = img1.detach().cpu().numpy()
    img2 = img2.detach().cpu().numpy()

    psnr_vals = []
    for i in range(img1.shape[0]):
        im1 = (img1[i].transpose(1, 2, 0) + 1) / 2
        im2 = (img2[i].transpose(1, 2, 0) + 1) / 2
        psnr_vals.append(psnr(im1, im2, data_range=1.0))

    return np.mean(psnr_vals)


def calculate_ssim(img1, img2):
    """Calculate SSIM between two images"""
    img1 = img1.detach().cpu().numpy()
    img2 = img2.detach().cpu().numpy()

    ssim_vals = []
    for i in range(img1.shape[0]):
        im1 = (img1[i].transpose(1, 2, 0) + 1) / 2
        im2 = (img2[i].transpose(1, 2, 0) + 1) / 2
        ssim_vals.append(ssim(im1, im2, data_range=1.0, channel_axis=2))

    return np.mean(ssim_vals)

## Cell 11: Training Stage 1 — Autoencoder (Paper §III.A)

Trains the Latent Autoencoder in isolation. Goal: learn to compress and reconstruct images so the latent space captures meaningful features.

**Loss:** $\mathcal{L}_{AE} = \|I_{HR} - D_\phi(E_\theta(I_{HR}))\|_1$

L1 loss is chosen over L2 because it produces sharper reconstructions — L2 tends to average over high-frequency details, resulting in blurring.

**Expected convergence (Paper §IV.B):** The autoencoder converges rapidly due to its deterministic pixel-based reconstruction objective. Loss should decrease steeply in the first 5 epochs and plateau around epoch 15–20.

**Line-by-line breakdown:**

- `train_autoencoder(model, dataloader, epochs=20, lr=1e-4)`:
  - **Optimizer** — `AdamW(model.autoencoder.parameters(), lr=1e-4, weight_decay=1e-4)`. Only autoencoder weights are updated.
  - **Scheduler** — `CosineAnnealingLR` with `T_max=epochs` for smooth LR decay.
  - **Training loop** — for each batch:
    1. Moves HR images to GPU.
    2. `model.autoencoder(hr)` → returns `(reconstruction, latent)`.
    3. Computes `L1(reconstruction, hr)`.
    4. Backward pass + optimizer step + scheduler step.
  - **MLflow** — `mlflow.log_metric("ae_train_loss", avg_loss, step=epoch)` logs loss to the active MLflow run.
  - **Checkpoint** — saves `autoencoder_best.pth` to the DBFS checkpoint directory after training.
  - Returns list of per-epoch losses for plotting.

In [ ]:
# Cell 12: Training Stage 1 - Autoencoder (NO mixed precision)
def train_autoencoder(model, train_loader, val_loader, num_epochs=20, lr=1e-4, device='cuda'):
    """Train the autoencoder (Stage 1) - NO mixed precision"""
    print("\n" + "="*50)
    print("STAGE 1: Training Autoencoder")
    print("="*50)

    model.autoencoder.to(device)
    optimizer = torch.optim.AdamW(model.autoencoder.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    l1_loss_fn = nn.L1Loss()
    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        model.autoencoder.train()
        train_losses = []

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for batch_idx, batch in enumerate(pbar):
            hr = batch['hr'].to(device)

            optimizer.zero_grad(set_to_none=True)

            recon, z = model.autoencoder(hr)
            loss = l1_loss_fn(recon, hr)

            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

            if batch_idx % 10 == 0:
                torch.cuda.empty_cache()

            del hr, recon, z, loss

        # Validation
        model.autoencoder.eval()
        val_losses = []
        val_psnr = []

        with torch.no_grad():
            for batch in val_loader:
                hr = batch['hr'].to(device)
                recon, z = model.autoencoder(hr)
                loss = l1_loss_fn(recon, hr)

                val_losses.append(loss.item())
                val_psnr.append(calculate_psnr(recon, hr))

                del hr, recon, z, loss
                torch.cuda.empty_cache()

        avg_train_loss = np.mean(train_losses)
        avg_val_loss = np.mean(val_losses)
        avg_val_psnr = np.mean(val_psnr)

        print(f"Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}, Val PSNR={avg_val_psnr:.2f}dB")

        # --- MLflow logging ---
        mlflow.log_metrics({
            "stage1_train_loss": avg_train_loss,
            "stage1_val_loss": avg_val_loss,
            "stage1_val_psnr": avg_val_psnr,
        }, step=epoch)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.autoencoder.state_dict(), AUTOENCODER_PATH)
            print(f"  Saved best autoencoder model")

        scheduler.step()

    # --- MLflow: log best metric ---
    mlflow.log_metric("stage1_best_val_loss", best_val_loss)
    mlflow.log_artifact(AUTOENCODER_PATH, artifact_path="checkpoints")

    print("\n  Stage 1 training completed!")
    return model

## Cell 12: Training Stage 2 — Neural Operator (Paper §III.B)

Trains the FNO and Implicit Amplification modules. The autoencoder is **frozen** (no gradients).

**Loss:** $\mathcal{L}_{NO} = \|z_{HR} - \hat{z}_{NO}\|_2^2$

where $z_{HR} = E_\theta(I_{HR})$ is the target latent (from frozen AE) and $\hat{z}_{NO} = \text{ImplicitAmp}(N_\psi(E_\theta(I_{LR}), s))$ is the predicted latent.

**Expected convergence (Paper §IV.B):** The neural operator stage exhibits gradual stabilization, capturing scale-invariant representations in the latent domain. Loss decreases more slowly than Stage 1 since it's learning in a compressed feature space.

**Line-by-line breakdown:**

- `train_neural_operator(model, dataloader, epochs=15, lr=1e-4)`:
  - **Freeze AE** — `model.autoencoder.eval()` + `param.requires_grad = False` for all AE parameters. This preserves the learned latent space.
  - **Optimizer** — `AdamW` updates only `neural_op` and `implicit_amp` parameters (combined via `itertools.chain`).
  - **Scheduler** — `CosineAnnealingLR` for smooth learning rate decay.
  - **Training loop** — for each batch:
    1. Encodes HR with frozen AE → target latent $z_{HR}$.
    2. Encodes LR with frozen AE → input latent $z_{LR}$.
    3. `neural_op(z_lr, scale_factor)` → $z_{refined}$ (frequency-domain enhancement).
    4. `implicit_amp(z_refined, scale_factor)` → $z_{amplified}$ (scale-conditioned modulation).
    5. Loss = MSE($z_{amplified}$, $z_{HR}$).
  - **MLflow** — logs `fno_train_loss` per epoch.
  - **Checkpoint** — saves `neural_operator_best.pth` (both neural_op and implicit_amp state dicts).

In [ ]:
# Cell 13: Training Stage 2 - Neural Operator (NO mixed precision - CRITICAL FIX)
def train_neural_operator(model, train_loader, val_loader, num_epochs=15, lr=1e-4, device='cuda'):
    """Train the Neural Operator (Stage 2) - NO mixed precision to avoid cuFFT errors"""
    print("\n" + "="*50)
    print("STAGE 2: Training Neural Operator")
    print("="*50)

    for param in model.autoencoder.parameters():
        param.requires_grad = False

    model.autoencoder.eval()
    model.neural_operator.to(device)

    optimizer = torch.optim.AdamW(model.neural_operator.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    mse_loss_fn = nn.MSELoss()
    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        model.neural_operator.train()
        train_losses = []

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for batch_idx, batch in enumerate(pbar):
            lr_img = batch['lr'].to(device)
            hr_img = batch['hr'].to(device)
            scale = batch['scale'][0].item()

            optimizer.zero_grad(set_to_none=True)

            with torch.no_grad():
                target_latent = model.encode_hr(hr_img)

            no_prior = model.get_no_prior(lr_img, scale)
            loss = mse_loss_fn(no_prior, target_latent)

            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

            if batch_idx % 10 == 0:
                torch.cuda.empty_cache()

            del lr_img, hr_img, target_latent, no_prior, loss

        # Validation
        model.neural_operator.eval()
        val_losses = []

        with torch.no_grad():
            for batch in val_loader:
                lr_img = batch['lr'].to(device)
                hr_img = batch['hr'].to(device)
                scale = batch['scale'][0].item()

                target_latent = model.encode_hr(hr_img)
                no_prior = model.get_no_prior(lr_img, scale)

                loss = mse_loss_fn(no_prior, target_latent)
                val_losses.append(loss.item())

                del lr_img, hr_img, target_latent, no_prior, loss
                torch.cuda.empty_cache()

        avg_train_loss = np.mean(train_losses)
        avg_val_loss = np.mean(val_losses)

        print(f"Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}")

        # --- MLflow logging ---
        mlflow.log_metrics({
            "stage2_train_loss": avg_train_loss,
            "stage2_val_loss": avg_val_loss,
        }, step=epoch)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.neural_operator.state_dict(), NEURAL_OPERATOR_PATH)
            print(f"  Saved best neural operator model")

        scheduler.step()

    # --- MLflow: log best metric ---
    mlflow.log_metric("stage2_best_val_loss", best_val_loss)
    mlflow.log_artifact(NEURAL_OPERATOR_PATH, artifact_path="checkpoints")

    print("\n  Stage 2 training completed!")
    return model

## Cell 13: Training Stage 3 — Diffusion UNet (Paper §III.C)

Trains the DiffusionUNet as the final refinement stage. Both the autoencoder and neural operator are **frozen**.

**Loss:** $\mathcal{L}_{Diff} = \mathbb{E}_{t, \epsilon}\left[\|\epsilon - \epsilon_\theta(x_t, t, c)\|_2^2\right]$

where $x_t$ is the noisy latent at timestep $t$, $\epsilon$ is the true Gaussian noise, $\epsilon_\theta$ is the UNet's prediction, and $c = z_{refined}$ is the neural operator context.

**Expected convergence (Paper §IV.B):** The diffusion stage demonstrates slower but smoother convergence, typical for probabilistic models. The loss should decrease gradually over 30 epochs with stable dynamics and reduced overfitting compared to GAN-based alternatives.

**Line-by-line breakdown:**

- `train_diffusion(model, dataloader, epochs=30, lr=1e-4)`:
  - **Freeze** — AE, neural_op, and implicit_amp are all set to `eval()` + `requires_grad = False`.
  - **Optimizer** — `AdamW` for only `diffusion_unet` parameters.
  - **Scheduler** — `CosineAnnealingLR` for smooth decay.
  - **Training loop** — for each batch:
    1. Frozen AE encodes HR → $z_{target}$.
    2. Frozen AE encodes LR → frozen neural_op → frozen implicit_amp → $z_{amplified}$.
    3. Sample random timesteps $t \sim \text{Uniform}\{0, ..., T-1\}$ and noise $\epsilon \sim \mathcal{N}(0, \mathbf{I})$.
    4. Forward diffusion: $x_t = \sqrt{\bar{\alpha}_t}\, z_{amplified} + \sqrt{1 - \bar{\alpha}_t}\, \epsilon$.
    5. UNet predicts: $\hat{\epsilon} = \text{UNet}(x_t, t, z_{refined})$.
    6. Loss = $\text{MSE}(\hat{\epsilon}, \epsilon)$.
  - `torch.cuda.empty_cache()` — called per batch to manage GPU memory (diffusion training is memory-intensive).
  - **MLflow** — logs `diff_train_loss` per epoch.
  - **Checkpoint** — saves `diffusion_best.pth`.

In [ ]:
# Cell 14: Training Stage 3 - Diffusion Model
def train_diffusion(model, train_loader, val_loader, num_epochs=30, lr=1e-4, device='cuda'):
    """Train the Diffusion Model (Stage 3) - Only trains DiffusionUNet"""
    print("\n" + "="*50)
    print("STAGE 3: Training Diffusion Model")
    print("="*50)

    # Freeze everything except diffusion_unet
    for param in model.autoencoder.parameters():
        param.requires_grad = False
    for param in model.neural_operator.parameters():
        param.requires_grad = False
    for param in model.implicit_amp.parameters():
        param.requires_grad = False

    model.autoencoder.eval()
    model.neural_operator.eval()
    model.implicit_amp.eval()
    model.diffusion_unet.to(device)

    optimizer = torch.optim.AdamW(model.diffusion_unet.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    mse_loss_fn = nn.MSELoss()
    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        model.diffusion_unet.train()
        train_losses = []

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for batch_idx, batch in enumerate(pbar):
            lr_img = batch['lr'].to(device)
            hr_img = batch['hr'].to(device)
            scale = batch['scale'][0].item()

            optimizer.zero_grad(set_to_none=True)

            with torch.no_grad():
                target_latent = model.encode_hr(hr_img)
                no_prior = model.get_no_prior(lr_img, scale)

                b = lr_img.shape[0]
                context = F.adaptive_avg_pool2d(no_prior, 1).view(b, -1)

            timesteps = torch.randint(0, model.scheduler.num_timesteps, (b,), device=device).long()
            noise = torch.randn_like(target_latent)
            noisy_latent = model.scheduler.add_noise(target_latent, noise, timesteps)

            noise_pred = model.diffusion_unet(noisy_latent, timesteps, context)

            loss = mse_loss_fn(noise_pred, noise)

            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

            if batch_idx % 10 == 0:
                torch.cuda.empty_cache()

            del lr_img, hr_img, target_latent, no_prior, context, timesteps, noise, noisy_latent, noise_pred, loss

        # Validation
        model.diffusion_unet.eval()
        val_losses = []

        with torch.no_grad():
            for batch in val_loader:
                lr_img = batch['lr'].to(device)
                hr_img = batch['hr'].to(device)
                scale = batch['scale'][0].item()

                target_latent = model.encode_hr(hr_img)
                no_prior = model.get_no_prior(lr_img, scale)

                b = lr_img.shape[0]
                context = F.adaptive_avg_pool2d(no_prior, 1).view(b, -1)

                timesteps = torch.randint(0, model.scheduler.num_timesteps, (b,), device=device).long()
                noise = torch.randn_like(target_latent)
                noisy_latent = model.scheduler.add_noise(target_latent, noise, timesteps)

                noise_pred = model.diffusion_unet(noisy_latent, timesteps, context)
                loss = mse_loss_fn(noise_pred, noise)

                val_losses.append(loss.item())

                del lr_img, hr_img, target_latent, no_prior, context, timesteps, noise, noisy_latent, noise_pred, loss
                torch.cuda.empty_cache()

        avg_train_loss = np.mean(train_losses)
        avg_val_loss = np.mean(val_losses)

        print(f"Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}")

        # --- MLflow logging ---
        mlflow.log_metrics({
            "stage3_train_loss": avg_train_loss,
            "stage3_val_loss": avg_val_loss,
        }, step=epoch)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                'diffusion_unet': model.diffusion_unet.state_dict(),
            }, DIFFUSION_PATH)
            print(f"  Saved best diffusion model")

        scheduler.step()

    # --- MLflow: log best metric ---
    mlflow.log_metric("stage3_best_val_loss", best_val_loss)
    mlflow.log_artifact(DIFFUSION_PATH, artifact_path="checkpoints")

    print("\n  Stage 3 training completed!")
    return model

## Cell 14: Evaluation & Comparison Grid (Paper §IV)

Contains two functions: `evaluate_model()` computes metrics on the test set, and `create_comparison_grid()` generates a visual side-by-side comparison.

**Expected stage-wise results (from Paper §IV.A):**

| Stage | PSNR (dB) | SSIM | LPIPS |
|-------|-----------|------|-------|
| Stage 1 — Autoencoder only | 27.68 | 0.81 | — |
| Stage 2 — + Neural Operator | 28.35 | 0.84 | — |
| Stage 3 — + Diffusion (full HNDSR) | **29.40** | **0.87** | **0.16** |

Each stage progressively improves quality: the AE captures coarse structures, the FNO adds global frequency-domain detail, and diffusion produces fine textures. The diffusion stage alone adds +1.05 dB PSNR over the FNO-only output.

**Line-by-line breakdown:**

- `evaluate_model(model, dataloader, num_batches=5)`:
  - Runs in `torch.no_grad()` mode (no gradient computation → saves memory).
  - For each batch, calls `model.super_resolve(lr)` to get SR output.
  - Denormalizes both SR and HR from [-1,1] to [0,1] via `(x + 1) / 2`.
  - Computes PSNR and SSIM for each image in the batch, accumulates averages.
  - Returns average PSNR and SSIM across all evaluated images.

- `create_comparison_grid(model, dataloader, num_images=4)`:
  - Takes the first `num_images` samples from the dataloader.
  - For each sample, creates a row: **LR** (bicubic upscaled to HR size) | **SR** (model output) | **HR** (ground truth).
  - Uses `F.interpolate(lr, size=hr.shape[-2:], mode='bicubic')` to resize LR for visual comparison.
  - Creates a matplotlib figure with a 3-column grid and saves it using `mlflow.log_figure()`.
  - Also saves individual SR images as artifacts via `mlflow.log_artifact()`.
  - Prints PSNR/SSIM per image for detailed analysis.

In [ ]:
# Cell 16: Evaluation Metrics and Visualization
def evaluate_model(model, test_loader, device='cuda', save_results=True, output_dir='results'):
    """
    Comprehensive evaluation with PSNR, SSIM, LPIPS metrics
    """
    import os
    from pathlib import Path

    # Create output directory
    if save_results:
        Path(output_dir).mkdir(exist_ok=True)
        Path(f"{output_dir}/lr").mkdir(exist_ok=True)
        Path(f"{output_dir}/sr").mkdir(exist_ok=True)
        Path(f"{output_dir}/hr").mkdir(exist_ok=True)

    # Initialize LPIPS
    try:
        lpips_fn = lpips.LPIPS(net='alex').to(device)
        use_lpips = True
    except:
        print("LPIPS not available, skipping perceptual metric")
        use_lpips = False

    model.autoencoder.eval()
    model.neural_operator.eval()
    model.implicit_amp.eval()
    model.diffusion_unet.eval()

    psnr_values = []
    ssim_values = []
    lpips_values = []

    print("\n" + "="*60)
    print("EVALUATING MODEL ON TEST SET")
    print("="*60)

    with torch.no_grad():
        for idx, batch in enumerate(tqdm(test_loader, desc="Evaluating")):
            lr_img = batch['lr'].to(device)
            hr_img = batch['hr'].to(device)
            scale = batch['scale'][0].item()

            sr_img = model.super_resolve(lr_img, scale_factor=scale, num_inference_steps=50)

            psnr_val = calculate_psnr(sr_img, hr_img)
            ssim_val = calculate_ssim(sr_img, hr_img)

            psnr_values.append(psnr_val)
            ssim_values.append(ssim_val)

            if use_lpips:
                lpips_val = lpips_fn(sr_img, hr_img).mean().item()
                lpips_values.append(lpips_val)

            if save_results and idx < 10:
                save_image((lr_img + 1) / 2, f"{output_dir}/lr/sample_{idx:03d}.png")
                save_image((sr_img + 1) / 2, f"{output_dir}/sr/sample_{idx:03d}.png")
                save_image((hr_img + 1) / 2, f"{output_dir}/hr/sample_{idx:03d}.png")

            del lr_img, hr_img, sr_img
            torch.cuda.empty_cache()

    # Compute statistics
    avg_psnr = np.mean(psnr_values)
    std_psnr = np.std(psnr_values)
    avg_ssim = np.mean(ssim_values)
    std_ssim = np.std(ssim_values)

    print("\n" + "="*60)
    print("FINAL RESULTS")
    print("="*60)
    print(f"PSNR: {avg_psnr:.2f} +/- {std_psnr:.2f} dB")
    print(f"SSIM: {avg_ssim:.4f} +/- {std_ssim:.4f}")

    if use_lpips:
        avg_lpips = np.mean(lpips_values)
        std_lpips = np.std(lpips_values)
        print(f"LPIPS: {avg_lpips:.4f} +/- {std_lpips:.4f}")

    print("="*60)

    if save_results:
        print(f"\nResults saved to '{output_dir}/' directory")

    results = {
        'psnr_mean': avg_psnr,
        'psnr_std': std_psnr,
        'ssim_mean': avg_ssim,
        'ssim_std': std_ssim,
        'psnr_values': psnr_values,
        'ssim_values': ssim_values,
    }

    if use_lpips:
        results['lpips_mean'] = avg_lpips
        results['lpips_std'] = std_lpips
        results['lpips_values'] = lpips_values

    torch.save(results, f"{output_dir}/evaluation_results.pth")
    print(f"Metrics saved to '{output_dir}/evaluation_results.pth'")

    return results


def create_comparison_grid(lr_path, sr_path, hr_path, output_path='comparison.png'):
    """
    Create side-by-side comparison of LR, SR, and HR images
    """
    from PIL import Image
    import matplotlib.pyplot as plt

    lr_img = Image.open(lr_path)
    sr_img = Image.open(sr_path)
    hr_img = Image.open(hr_path)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(lr_img)
    axes[0].set_title('Low Resolution (Input)', fontsize=14)
    axes[0].axis('off')

    axes[1].imshow(sr_img)
    axes[1].set_title('Super-Resolved (Ours)', fontsize=14)
    axes[1].axis('off')

    axes[2].imshow(hr_img)
    axes[2].set_title('High Resolution (Ground Truth)', fontsize=14)
    axes[2].axis('off')

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()

    print(f"Comparison saved to '{output_path}'")

## Cell 15: MLflow-Tracked Training Pipeline

The main entry point that orchestrates all three training stages within a single MLflow experiment run.

This implements the **sequential training strategy** from Paper §III: each stage is trained independently with its own optimizer and loss function, while freezing all previously trained components. This avoids gradient interference between the deterministic (AE, FNO) and stochastic (diffusion) pathways.

**Effective total loss (Paper §III.D):**

$$\mathcal{L}_{total} = \underbrace{\|I_{HR} - \hat{I}_{HR}\|_1}_{\text{Stage 1}} + \lambda_1 \underbrace{\|z_{HR} - \hat{z}_{NO}\|_2^2}_{\text{Stage 2}} + \lambda_2 \underbrace{\mathbb{E}[\|\epsilon - \epsilon_\theta\|_2^2]}_{\text{Stage 3}}$$

**Line-by-line breakdown:**

- `main_training_pipeline_mlflow()`:
  - `mlflow.set_experiment("/HNDSR-Super-Resolution")` — creates or sets the MLflow experiment.
  - `mlflow.start_run(run_name=...)` — starts a tracked run with a timestamped name.
  - **Log hyperparameters** — `mlflow.log_params(...)` records all key hyperparameters: `ae_latent_dim`, `fno_modes`, `fno_width`, `diffusion_channels`, `num_timesteps`, `patch_size`, `batch_size`, learning rates, and epoch counts.
  - **Model creation** — `HNDSR().to(device)` instantiates the full model and logs total parameter count.
  - **Sequential training** — calls `train_autoencoder()` → `train_neural_operator()` → `train_diffusion()` in order, each logging its own loss curve to MLflow.
  - **Evaluation** — calls `evaluate_model()` and logs final `test_psnr` and `test_ssim` as summary metrics.
  - **Comparison grid** — `create_comparison_grid()` generates and logs visual artifacts.
  - **Model registry** — `mlflow.pytorch.log_model(model, "hndsr_model", registered_model_name="HNDSR-v1")` saves the complete model to MLflow's Model Registry for versioned deployment.
  - **Final checkpoint** — saves `hndsr_complete.pth` with all component state dicts to DBFS.
  - Returns the trained model.

In [ ]:
# Cell 15: MLflow-Tracked Training Pipeline
def main_training_pipeline_mlflow(hr_dir, lr_dir, batch_size=2, num_workers=0):
    """Complete training pipeline for HNDSR with full MLflow tracking."""

    torch.cuda.empty_cache()
    gc.collect()

    # ---- Dataset ----
    print("Loading datasets...")
    try:
        train_dataset = SatelliteDataset(hr_dir, lr_dir, patch_size=64, training=True)
    except ValueError as e:
        print(f"\nERROR: {e}")
        return None

    if len(train_dataset) < 10:
        print(f"\nWARNING: Only {len(train_dataset)} images found!")
        return None

    train_size = int(0.9 * len(train_dataset))
    val_size = len(train_dataset) - train_size

    if train_size == 0 or val_size == 0:
        print(f"\nERROR: Not enough data! Total images: {len(train_dataset)}")
        return None

    train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                             num_workers=num_workers, pin_memory=False, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                           num_workers=num_workers, pin_memory=False)

    print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

    # ---- Model ----
    print("\nInitializing HNDSR model...")
    model = HNDSR(
        ae_latent_dim=128,
        ae_downsample_ratio=8,
        no_width=32,
        no_modes=8,
        diffusion_channels=64,
        num_timesteps=1000
    )

    # ================================================================
    # MLflow Parent Run – wraps all three training stages
    # ================================================================
    with mlflow.start_run(run_name="HNDSR_Full_Training") as parent_run:

        # Log hyperparameters once
        mlflow.log_params({
            "dataset_hr": hr_dir,
            "dataset_lr": lr_dir,
            "batch_size": batch_size,
            "patch_size": 64,
            "ae_latent_dim": 128,
            "ae_downsample_ratio": 8,
            "no_width": 32,
            "no_modes": 8,
            "diffusion_channels": 64,
            "num_timesteps": 1000,
            "stage1_epochs": 20,
            "stage1_lr": 1e-4,
            "stage2_epochs": 15,
            "stage2_lr": 1e-4,
            "stage3_epochs": 30,
            "stage3_lr": 1e-4,
            "optimizer": "AdamW",
            "scheduler": "CosineAnnealingLR",
            "seed": 42,
        })

        # Tag the run for easy filtering in the Databricks UI
        mlflow.set_tags({
            "model": "HNDSR",
            "task": "satellite_super_resolution",
            "framework": "pytorch",
        })

        # ------- Stage 1 -------
        model = train_autoencoder(model, train_loader, val_loader,
                                 num_epochs=20, lr=1e-4, device=device)
        torch.cuda.empty_cache()
        gc.collect()

        # ------- Stage 2 -------
        model = train_neural_operator(model, train_loader, val_loader,
                                      num_epochs=15, lr=1e-4, device=device)
        torch.cuda.empty_cache()
        gc.collect()

        # ------- Stage 3 -------
        model = train_diffusion(model, train_loader, val_loader,
                               num_epochs=30, lr=1e-4, device=device)

        # ------- Save complete model -------
        torch.save({
            'autoencoder': model.autoencoder.state_dict(),
            'neural_operator': model.neural_operator.state_dict(),
            'diffusion_unet': model.diffusion_unet.state_dict(),
            'implicit_amp': model.implicit_amp.state_dict(),
        }, COMPLETE_MODEL_PATH)

        mlflow.log_artifact(COMPLETE_MODEL_PATH, artifact_path="checkpoints")
        print(f"\nTraining complete! Model saved to {COMPLETE_MODEL_PATH}")

        # ------- Automatic Evaluation -------
        try:
            print("\nStarting automatic evaluation...")
            test_dataset = SatelliteDataset(hr_dir, lr_dir, patch_size=64, training=False)
            num_samples = min(50, len(test_dataset))
            if len(test_dataset) > num_samples:
                indices = np.random.choice(len(test_dataset), num_samples, replace=False)
                test_dataset = torch.utils.data.Subset(test_dataset, indices)
            test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)

            results = evaluate_model(model, test_loader, device,
                                    save_results=True, output_dir=EVAL_RESULTS_DIR)

            # Log final evaluation metrics to MLflow
            mlflow.log_metrics({
                "final_psnr_mean": results['psnr_mean'],
                "final_psnr_std": results['psnr_std'],
                "final_ssim_mean": results['ssim_mean'],
                "final_ssim_std": results['ssim_std'],
            })
            if 'lpips_mean' in results:
                mlflow.log_metrics({
                    "final_lpips_mean": results['lpips_mean'],
                    "final_lpips_std": results['lpips_std'],
                })

            # Log evaluation artifacts (sample images)
            mlflow.log_artifacts(EVAL_RESULTS_DIR, artifact_path="evaluation")

            print("\n" + "="*70)
            print("EVERYTHING COMPLETE!")
            print("="*70)
            print(f"\nFinal Metrics:")
            print(f"   PSNR : {results['psnr_mean']:.2f} +/- {results['psnr_std']:.2f} dB")
            print(f"   SSIM : {results['ssim_mean']:.4f} +/- {results['ssim_std']:.4f}")
            if 'lpips_mean' in results:
                print(f"   LPIPS: {results['lpips_mean']:.4f} +/- {results['lpips_std']:.4f}")

        except Exception as e:
            print(f"\nEvaluation failed: {e}")
            print("   You can run the evaluation cell manually.")

        # ------- Register model in MLflow Model Registry -------
        try:
            mlflow.pytorch.log_model(
                pytorch_model=model,
                artifact_path="hndsr_model",
                registered_model_name=REGISTERED_MODEL_NAME,
            )
            print(f"\nModel registered as '{REGISTERED_MODEL_NAME}' in MLflow Model Registry")
        except Exception as e:
            print(f"\nModel registration skipped: {e}")

    return model


# ========================================================
# RUN THE PIPELINE
# ========================================================
print(f"Using paths:")
print(f"   HR: {HR_DIR}")
print(f"   LR: {LR_DIR}")
print(f"\nStarting MLflow-tracked training...")

model = main_training_pipeline_mlflow(
    hr_dir=HR_DIR,
    lr_dir=LR_DIR,
    batch_size=2,
    num_workers=0
)

## Cell 16: Load Checkpoints & Re-evaluate

Utility to reload a previously trained model from DBFS checkpoints and run evaluation without retraining.

**Line-by-line breakdown:**

- `load_complete_model(checkpoint_dir)`:
  - Creates a fresh `HNDSR()` instance.
  - Loads each component's `state_dict` from the corresponding `.pth` file (`autoencoder_best.pth`, `neural_operator_best.pth`, `diffusion_best.pth`).
  - Returns the fully loaded model on the specified device.

- `run_final_evaluation()`:
  - Calls `load_complete_model()` to restore weights.
  - Runs `evaluate_model()` on the test dataloader.
  - Calls `create_comparison_grid()` to generate visual comparison.
  - Prints final PSNR and SSIM.
  - Useful for re-evaluating after a cluster restart without retraining.

In [ ]:
# Cell 17: Load Trained Model and Evaluate
def load_complete_model(checkpoint_path, device='cuda'):
    """Load the complete trained model from checkpoint"""
    print("\n" + "="*60)
    print("LOADING COMPLETE MODEL")
    print("="*60)

    model = HNDSR(
        ae_latent_dim=128,
        ae_downsample_ratio=8,
        no_width=32,
        no_modes=8,
        diffusion_channels=64,
        num_timesteps=1000
    )

    checkpoint = torch.load(checkpoint_path, map_location=device)

    model.autoencoder.load_state_dict(checkpoint['autoencoder'])
    model.neural_operator.load_state_dict(checkpoint['neural_operator'])
    model.diffusion_unet.load_state_dict(checkpoint['diffusion_unet'])
    model.implicit_amp.load_state_dict(checkpoint['implicit_amp'])

    model.autoencoder.to(device)
    model.neural_operator.to(device)
    model.implicit_amp.to(device)
    model.diffusion_unet.to(device)

    print("Model loaded successfully!")
    print("="*60)
    return model


def run_final_evaluation(hr_dir, lr_dir, checkpoint_path,
                         batch_size=1, num_samples=50, device='cuda'):
    """Run complete evaluation pipeline after training"""
    print("\n" + "="*70)
    print("FINAL MODEL EVALUATION")
    print("="*70)

    model = load_complete_model(checkpoint_path, device)

    print("\nLoading test dataset...")
    test_dataset = SatelliteDataset(hr_dir, lr_dir, patch_size=64, training=False)

    if len(test_dataset) > num_samples:
        indices = np.random.choice(len(test_dataset), num_samples, replace=False)
        test_dataset = torch.utils.data.Subset(test_dataset, indices)

    test_loader = DataLoader(test_dataset, batch_size=batch_size,
                             shuffle=False, num_workers=0)

    print(f"Test samples: {len(test_dataset)}")

    results = evaluate_model(model, test_loader, device,
                            save_results=True, output_dir=EVAL_RESULTS_DIR)

    # Create comparison grids
    print("\nCreating comparison visualizations...")
    for i in range(min(5, len(test_dataset))):
        create_comparison_grid(
            f'{EVAL_RESULTS_DIR}/lr/sample_{i:03d}.png',
            f'{EVAL_RESULTS_DIR}/sr/sample_{i:03d}.png',
            f'{EVAL_RESULTS_DIR}/hr/sample_{i:03d}.png',
            f'{EVAL_RESULTS_DIR}/comparison_{i:03d}.png'
        )

    print("\n" + "="*70)
    print("EVALUATION COMPLETE!")
    print("="*70)
    print(f"\nSummary:")
    print(f"   PSNR : {results['psnr_mean']:.2f} +/- {results['psnr_std']:.2f} dB")
    print(f"   SSIM : {results['ssim_mean']:.4f} +/- {results['ssim_std']:.4f}")
    if 'lpips_mean' in results:
        print(f"   LPIPS: {results['lpips_mean']:.4f} +/- {results['lpips_std']:.4f}")
    print(f"\nResults saved in '{EVAL_RESULTS_DIR}/' directory")

    return results


# Uncomment to evaluate a saved checkpoint:
# results = run_final_evaluation(
#     hr_dir=HR_DIR,
#     lr_dir=LR_DIR,
#     checkpoint_path=COMPLETE_MODEL_PATH,
#     batch_size=1,
#     num_samples=50,
#     device=device
# )

## Cell 17: Query MLflow Runs

Searches the MLflow experiment for past runs, allowing you to compare hyperparameters and results across experiments.

**Line-by-line breakdown:**

- `mlflow.set_experiment("/HNDSR-Super-Resolution")` — points to the correct experiment.
- `mlflow.search_runs(order_by=["start_time DESC"])` — retrieves all runs sorted by most recent first. Returns a Pandas DataFrame.
- `runs[["run_id", "status", ...]]` — selects key columns: run ID, status, PSNR, SSIM, and hyperparameters for a concise comparison table.
- Use this after multiple training runs to identify which hyperparameter configuration produced the best results.

In [ ]:
# Cell 17: Query MLflow Experiment Results
#
# On Databricks you can also view results in the Experiments sidebar,
# but this cell lets you query programmatically.

import mlflow

# Show all runs in the experiment, ordered by PSNR
runs_df = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=["metrics.final_psnr_mean DESC"],
)

# Display key columns
cols = [
    "run_id", "run_name", "status",
    "metrics.final_psnr_mean", "metrics.final_ssim_mean", "metrics.final_lpips_mean",
    "metrics.stage1_best_val_loss", "metrics.stage2_best_val_loss", "metrics.stage3_best_val_loss",
]
available_cols = [c for c in cols if c in runs_df.columns]
display(runs_df[available_cols])

## Cell 18: Load Model from MLflow Registry

Loads a trained model directly from the MLflow Model Registry by name and version, useful for deployment or further evaluation.

**Line-by-line breakdown:**

- `model_name = "HNDSR-v1"` — the registered model name (set during `log_model()`).
- `model_version = 1` — loads version 1; increment for subsequent training runs.
- `model_uri = f"models:/{model_name}/{model_version}"` — MLflow URI format for the registry.
- `mlflow.pytorch.load_model(model_uri)` — downloads the model artifact and deserializes it via `torch.load()`.
- `.to(device).eval()` — moves to GPU and sets to evaluation mode (disables dropout/batchnorm training behavior).

In [ ]:
# Cell 18: Load Model from MLflow Registry for Inference
#
# After training and registration, you can load the model directly
# from the MLflow Model Registry for inference or deployment.

import mlflow.pytorch

# Load the latest version from the registry
model_uri = f"models:/{REGISTERED_MODEL_NAME}/latest"
print(f"Loading model from: {model_uri}")

try:
    loaded_model = mlflow.pytorch.load_model(model_uri)
    loaded_model.eval()
    loaded_model.to(device)
    print(f"Model loaded successfully from MLflow Registry!")
    print(f"Model is on device: {device}")
    print(f"\nReady for inference. Example:")
    print(f"   sr_image = loaded_model.super_resolve(lr_image, scale_factor=4)")
except Exception as e:
    print(f"Could not load from registry: {e}")
    print(f"Make sure you have run the training pipeline first.")

## Cell 19: Training Loss Curves from MLflow History

Retrieves per-epoch loss values from the MLflow run and plots loss curves for all three training stages.

**Line-by-line breakdown:**

- `client = mlflow.tracking.MlflowClient()` — programmatic access to the MLflow API.
- `runs = client.search_runs(...)` — finds the most recent completed run in the HNDSR experiment.
- `client.get_metric_history(run_id, "ae_train_loss")` — fetches all logged values for the autoencoder loss (one per epoch).
- `plt.subplot(1,3,i)` — creates a 3-panel figure: AE loss | FNO loss | Diffusion loss.
- Each subplot shows loss vs. epoch, revealing convergence behavior.
- `plt.savefig()` + `mlflow.log_artifact()` — saves the plot both locally and as an MLflow artifact.
- Use this to diagnose training: if loss is still decreasing, more epochs may help; if it plateaus, the stage has converged.

In [ ]:
# Cell 19: Plot training curves from MLflow metrics
import mlflow
import matplotlib.pyplot as plt

# Get the most recent run
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=["start_time DESC"],
    max_results=1,
)

if len(runs) == 0:
    print("No runs found. Train the model first.")
else:
    run_id = runs.iloc[0]["run_id"]
    client = mlflow.tracking.MlflowClient()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Stage 1
    s1_train = client.get_metric_history(run_id, "stage1_train_loss")
    s1_val   = client.get_metric_history(run_id, "stage1_val_loss")
    if s1_train:
        axes[0].plot([m.step for m in s1_train], [m.value for m in s1_train], label="Train Loss")
        axes[0].plot([m.step for m in s1_val],   [m.value for m in s1_val],   label="Val Loss")
        axes[0].set_title("Stage 1: Autoencoder")
        axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].grid(True)

    # Stage 2
    s2_train = client.get_metric_history(run_id, "stage2_train_loss")
    s2_val   = client.get_metric_history(run_id, "stage2_val_loss")
    if s2_train:
        axes[1].plot([m.step for m in s2_train], [m.value for m in s2_train], label="Train Loss")
        axes[1].plot([m.step for m in s2_val],   [m.value for m in s2_val],   label="Val Loss")
        axes[1].set_title("Stage 2: Neural Operator")
        axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(True)

    # Stage 3
    s3_train = client.get_metric_history(run_id, "stage3_train_loss")
    s3_val   = client.get_metric_history(run_id, "stage3_val_loss")
    if s3_train:
        axes[2].plot([m.step for m in s3_train], [m.value for m in s3_train], label="Train Loss")
        axes[2].plot([m.step for m in s3_val],   [m.value for m in s3_val],   label="Val Loss")
        axes[2].set_title("Stage 3: Diffusion Model")
        axes[2].set_xlabel("Epoch"); axes[2].legend(); axes[2].grid(True)

    plt.tight_layout()
    plt.show()

    # Also log the figure as an artifact
    fig.savefig("/tmp/training_curves.png", dpi=150, bbox_inches="tight")
    mlflow.log_artifact("/tmp/training_curves.png", artifact_path="plots")
    print("Training curves logged to MLflow artifacts.")

## Cell 20: Test Image — Before & After Inference (Paper §IV.D: Qualitative Analysis)

Visualizes a single test image through the full HNDSR pipeline, showing:
1. **LR Input** — the low-resolution image as received by the model
2. **Bicubic Upscale** — naive 4× bicubic interpolation (baseline)
3. **HNDSR Super-Resolved** — output of our model
4. **HR Ground Truth** — the original high-resolution image

From the paper: *"HNDSR reconstructs sharper edges, natural vegetation textures, and fine terrain variations, minimizing blurring artifacts commonly observed in regression-based networks."* The bicubic baseline serves as a lower bound — any ML-based SR method should outperform it on both PSNR and SSIM.

In [ ]:
# Cell 20: Test Image Before & After Inference
import matplotlib.pyplot as plt
import torch.nn.functional as F

def show_before_after(model, dataloader, sample_index=0, num_inference_steps=50):
    """
    Displays a single test image before and after HNDSR inference.
    
    Args:
        model: Trained HNDSR model
        dataloader: DataLoader with test images
        sample_index: Which image from the first batch to display (default: 0)
        num_inference_steps: Number of diffusion denoising steps (default: 50)
    """
    model.eval()
    device = next(model.parameters()).device
    
    # Get a batch from the dataloader
    batch = next(iter(dataloader))
    lr = batch['lr'].to(device)
    hr = batch['hr'].to(device)
    
    # Select one sample
    lr_single = lr[sample_index:sample_index+1]
    hr_single = hr[sample_index:sample_index+1]
    
    # Run inference
    with torch.no_grad():
        sr_single = model.super_resolve(lr_single, num_steps=num_inference_steps)
    
    # Denormalize from [-1, 1] to [0, 1]
    lr_display = (lr_single[0].cpu() + 1) / 2
    hr_display = (hr_single[0].cpu() + 1) / 2
    sr_display = (sr_single[0].cpu() + 1) / 2
    
    # Clamp to valid range
    lr_display = lr_display.clamp(0, 1)
    hr_display = hr_display.clamp(0, 1)
    sr_display = sr_display.clamp(0, 1)
    
    # Bicubic upscale of LR for comparison
    lr_upscaled = F.interpolate(
        lr_single, size=hr_single.shape[-2:], mode='bicubic', align_corners=False
    )
    lr_upscaled_display = ((lr_upscaled[0].cpu() + 1) / 2).clamp(0, 1)
    
    # Convert to numpy (H, W, C) for matplotlib
    lr_np = lr_display.permute(1, 2, 0).numpy()
    lr_up_np = lr_upscaled_display.permute(1, 2, 0).numpy()
    sr_np = sr_display.permute(1, 2, 0).numpy()
    hr_np = hr_display.permute(1, 2, 0).numpy()
    
    # Compute metrics
    psnr_bicubic = calculate_psnr(lr_upscaled_display, hr_display)
    ssim_bicubic = calculate_ssim(lr_upscaled_display, hr_display)
    psnr_sr = calculate_psnr(sr_display, hr_display)
    ssim_sr = calculate_ssim(sr_display, hr_display)
    
    # --- Plot 1: Simple Before & After ---
    fig1, axes1 = plt.subplots(1, 2, figsize=(10, 5))
    fig1.suptitle('HNDSR: Before & After Inference', fontsize=16, fontweight='bold')
    
    axes1[0].imshow(lr_np)
    axes1[0].set_title(f'Before (LR Input)\n{lr_np.shape[0]}×{lr_np.shape[1]}', fontsize=12)
    axes1[0].axis('off')
    
    axes1[1].imshow(sr_np)
    axes1[1].set_title(f'After (HNDSR Output)\nPSNR: {psnr_sr:.2f} dB | SSIM: {ssim_sr:.4f}', fontsize=12)
    axes1[1].axis('off')
    
    plt.tight_layout()
    plt.savefig('/tmp/hndsr_before_after.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # --- Plot 2: Full 4-panel comparison ---
    fig2, axes2 = plt.subplots(1, 4, figsize=(20, 5))
    fig2.suptitle('Full Comparison: LR → Bicubic → HNDSR → Ground Truth', fontsize=16, fontweight='bold')
    
    axes2[0].imshow(lr_np)
    axes2[0].set_title(f'LR Input\n({lr_np.shape[0]}×{lr_np.shape[1]})', fontsize=11)
    axes2[0].axis('off')
    
    axes2[1].imshow(lr_up_np)
    axes2[1].set_title(f'Bicubic 4× Upscale\nPSNR: {psnr_bicubic:.2f} dB | SSIM: {ssim_bicubic:.4f}', fontsize=11)
    axes2[1].axis('off')
    
    axes2[2].imshow(sr_np)
    axes2[2].set_title(f'HNDSR Output\nPSNR: {psnr_sr:.2f} dB | SSIM: {ssim_sr:.4f}', fontsize=11)
    axes2[2].axis('off')
    
    axes2[3].imshow(hr_np)
    axes2[3].set_title(f'Ground Truth (HR)\n({hr_np.shape[0]}×{hr_np.shape[1]})', fontsize=11)
    axes2[3].axis('off')
    
    plt.tight_layout()
    plt.savefig('/tmp/hndsr_full_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Print summary
    print("\n" + "="*60)
    print("IMAGE QUALITY COMPARISON")
    print("="*60)
    print(f"{'Method':<25} {'PSNR (dB)':<15} {'SSIM':<10}")
    print("-"*60)
    print(f"{'Bicubic Interpolation':<25} {psnr_bicubic:<15.2f} {ssim_bicubic:<10.4f}")
    print(f"{'HNDSR (Ours)':<25} {psnr_sr:<15.2f} {ssim_sr:<10.4f}")
    print("-"*60)
    improvement_psnr = psnr_sr - psnr_bicubic
    improvement_ssim = ssim_sr - ssim_bicubic
    print(f"{'Improvement':<25} {improvement_psnr:+<14.2f}  {improvement_ssim:+<9.4f}")
    print("="*60)

# --- Run the visualization ---
# Option 1: Use the model from training (if you just trained)
# show_before_after(model, dataloader)

# Option 2: Load from checkpoint and visualize
try:
    # Try to use 'model' if it exists from training
    show_before_after(model, dataloader, sample_index=0, num_inference_steps=50)
except NameError:
    # If model not in memory, load from checkpoint
    print("Loading model from checkpoints...")
    loaded_model = load_complete_model(CHECKPOINT_DIR)
    show_before_after(loaded_model, dataloader, sample_index=0, num_inference_steps=50)

## Cell 21: Multiple Test Samples — Before & After Grid

Visualizes several test images at once in a grid layout for a broader qualitative assessment. Each row shows one test image: LR input (before) → HNDSR output (after) → HR ground truth.

In [ ]:
# Cell 21: Multiple Test Samples Grid
def show_multiple_before_after(model, dataloader, num_samples=4, num_inference_steps=50):
    """
    Displays a grid of test images: LR (before) | SR (after) | HR (ground truth).
    
    Args:
        model: Trained HNDSR model
        dataloader: DataLoader with test images
        num_samples: Number of test images to display
        num_inference_steps: Diffusion sampling steps
    """
    model.eval()
    device = next(model.parameters()).device
    
    batch = next(iter(dataloader))
    lr = batch['lr'].to(device)
    hr = batch['hr'].to(device)
    
    n = min(num_samples, lr.shape[0])
    
    fig, axes = plt.subplots(n, 3, figsize=(15, 5 * n))
    fig.suptitle('HNDSR Test Results: Before → After → Ground Truth', fontsize=18, fontweight='bold', y=1.02)
    
    if n == 1:
        axes = axes.reshape(1, -1)
    
    col_titles = ['LR Input (Before)', 'HNDSR Output (After)', 'HR Ground Truth']
    for j, title in enumerate(col_titles):
        axes[0, j].set_title(title, fontsize=14, fontweight='bold', pad=10)
    
    for i in range(n):
        with torch.no_grad():
            sr = model.super_resolve(lr[i:i+1], num_steps=num_inference_steps)
        
        # Denormalize and clamp
        lr_img = ((lr[i].cpu() + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy()
        sr_img = ((sr[0].cpu() + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy()
        hr_img = ((hr[i].cpu() + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy()
        
        # Compute per-image metrics
        sr_t = ((sr[0].cpu() + 1) / 2).clamp(0, 1)
        hr_t = ((hr[i].cpu() + 1) / 2).clamp(0, 1)
        psnr = calculate_psnr(sr_t, hr_t)
        ssim = calculate_ssim(sr_t, hr_t)
        
        axes[i, 0].imshow(lr_img)
        axes[i, 0].set_ylabel(f'Sample {i+1}', fontsize=12, fontweight='bold', rotation=0, labelpad=60)
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(sr_img)
        axes[i, 1].set_xlabel(f'PSNR: {psnr:.2f} dB | SSIM: {ssim:.4f}', fontsize=10)
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(hr_img)
        axes[i, 2].axis('off')
        
        print(f"Sample {i+1}: PSNR = {psnr:.2f} dB, SSIM = {ssim:.4f}")
    
    plt.tight_layout()
    plt.savefig('/tmp/hndsr_multi_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

# --- Run multi-sample visualization ---
try:
    show_multiple_before_after(model, dataloader, num_samples=4, num_inference_steps=50)
except NameError:
    print("Loading model from checkpoints...")
    loaded_model = load_complete_model(CHECKPOINT_DIR)
    show_multiple_before_after(loaded_model, dataloader, num_samples=4, num_inference_steps=50)